# Evaluador de Tweets por Rúbrica

Califica cada tweet de un CSV contra cada criterio de una rúbrica en PDF, usando **agentes en paralelo de Google ADK**, y devuelve el CSV original enriquecido con nivel, puntaje y justificación por criterio.

Inspirado en la arquitectura de `Ajuste Evaluador`, adaptado a la inversión de escala: en lugar de un documento largo por corrida, miles de documentos cortos.

## Flujo

```
PDF rúbrica ──▶ RubricParserAgent ──▶ rubrica.json  👁 revisable
                                            │
                                            │  N = len(criterios)
CSV tweets ──▶ subconjunto ──▶ payload ──▶ ParallelAgent(N agentes)
                                            │
                                   checkpoint JSONL (reanudable)
                                            │
                              CSV enriquecido + CSV tidy ──▶ descarga
```

## Parámetros principales

Todos viven en la celda **CONFIG**. Los que más importan:

| Parámetro | Efecto |
|---|---|
| `MODEL_ID` | `gemini-2.5-flash`. El razonamiento se puede apagar en la familia 2.5; en 3.7-flash el piso es `low`. |
| `THINKING_BUDGET` | `0` = apagado. Dejarlo en automático multiplica el costo por ~2.7. |
| `N_MUESTRA` | Filas a evaluar. `None` = corpus completo. Empieza chico. |
| `JUSTIF_MAX_PALABRAS` | **La palanca de costo.** La salida domina el gasto. |
| `CONCURRENCIA` | Filas en vuelo simultáneo. Súbelo si no topas límites de tasa. |

## Costo

`costo ≈ N_criterios × $1.51` para 2,000 tweets · `× $11.11` para 14,635.
La celda de estimación lo calcula con tokens reales antes de gastar y pide confirmación.

## Limitaciones conocidas

- **Replies sin contexto.** Miles de filas son respuestas cuyo tweet padre no está en el corpus. Se marcan y se advierte al modelo, pero no se reconstruye el hilo. Filtra por `contexto_incompleto` en el análisis.
- **`aplicable` no es lo mismo que nivel bajo.** Un tweet ajeno al criterio se marca `aplicable=False` con nivel vacío. No lo promedies como cero.
- **Sin contraste humano por defecto.** Hasta que no cargues un gold set anotado a mano (§6), las calificaciones no tienen validación externa.
- **La rúbrica puede no discriminar** sobre tweets si fue concebida para otro tipo de texto. El histograma de §6 lo revela con un gasto de ~1 dólar.
- **Contenido ofensivo.** El corpus lo tiene. Los filtros de seguridad de Gemini pueden bloquear respuestas; se registran como `BLOQUEADO`, distinto de un nivel bajo.

In [ ]:
#@title 1 · Instalación de dependencias
# ADK 1.25.0 es la versión validada en el proyecto de referencia.
# openpyxl: pandas.read_excel lo necesita para el .xlsx del corpus.
!pip install -q "google-adk>=1.25.0" "google-genai>=1.63.0" pandas nest_asyncio openpyxl

import google.adk, google.genai
print("google-adk  ", google.adk.__version__ if hasattr(google.adk, "__version__") else "instalado")
print("google-genai", google.genai.__version__)

In [ ]:
#@title 2 · CONFIG — todos los parámetros en un solo lugar
from pathlib import Path

# ── Modelo ────────────────────────────────────────────────────────────────
MODEL_ID            = "gemini-2.5-flash"  # familia 2.5: el thinking se puede apagar
TEMPERATURE         = 0.0                 # 0 = reproducible (Ajuste Evaluador usa 0.2 por el voting)
THINKING_BUDGET     = 0                   # 0 = DESACTIVADO. -1 = automático (≈2.7× más caro)
MODEL_ID_CALIBRACION = "gemini-2.5-flash-lite"  # segundo modelo para §6.4

# ── Secreto ───────────────────────────────────────────────────────────────
SECRET_NAME         = "GOOGLE_API_KEY"    # nombre en el panel 🔑 de Colab

# ── Corpus ────────────────────────────────────────────────────────────────
# El archivo llega dañado en cuatro frentes: mojibake (UTF-8 leído como cp1252),
# identificadores de 19 dígitos redondeados por float64, 99 textos truncados a
# 255 caracteres y 458 con los saltos de línea colapsados. La celda 8.5 lo
# reconstruye desde 'raw_json', que sobrevivió íntegro.
# El .csv equivalente NO sirve: sufrió una segunda ronda que sustituyó por '?'
# los caracteres no representables, dentro de su propio raw_json. Es pérdida
# irreversible, y como las columnas planas de ambos son la misma exportación
# fila por fila, el CSV bueno se REGENERA desde el .xlsx.
CORPUS_FUENTE_XLSX  = "20260910 turistas_seleccionados.xlsx"
CORPUS_REPARADO     = "turistas_reparado.csv"
CORPUS_TRADUCIDO    = "turistas_traducido.csv"
FORZAR_REPARACION   = False        # True = rehacer aunque el reparado ya exista

# Se carga el traducido si existe; si no, el reparado. En una sesión limpia no
# existe ninguno todavía: la celda 8.5 crea el reparado antes de la carga.
CSV_PATH            = CORPUS_TRADUCIDO if Path(CORPUS_TRADUCIDO).exists() else CORPUS_REPARADO

COL_ID              = "id"
COL_TEXTO           = "text_completo"
COL_TRADUCCION      = "traducción"
COL_LANG            = "lang"

N_MUESTRA           = None         # None = corpus completo (2,562 filas ≈ $6.80)
                                   # Ponlo en 200 para una pasada barata de prueba.
SEED                = 42           # muestreo reproducible
EXCLUIR_RETWEETS    = False
EXCLUIR_QUOTES      = False
FILTRO_LANG         = None         # p.ej. ["es", "en"]; None = todos
FILTRO_TIPO_QUERY   = None         # p.ej. ["VICTIMIZACION"]; None = todos

# ── Rúbrica ───────────────────────────────────────────────────────────────
RUBRICA_PDF_PATH    = None         # None = subir en la celda de carga
RUBRICA_JSON_PATH   = "rubrica.json"

# ── Traducción  (celdas 9.5 y 9.6) ────────────────────────────────────────
# La traducción es para CONSUMO HUMANO: el objeto evaluado sigue siendo el texto
# original (regla 5 de instruccion_para). Se paga una sola vez sobre el corpus
# completo y se persiste; ninguna corrida posterior la vuelve a pagar.
MODEL_ID_TRADUCCION   = "gemini-2.5-flash-lite"
CKPT_TRADUCCION       = "checkpoint_traduccion.jsonl"
CONCURRENCIA_TRAD     = 8
RETRADUCIR_EXISTENTES = False      # True = sobrescribe las traducciones ya presentes

# ── Calificación ──────────────────────────────────────────────────────────
JUSTIF_MAX_PALABRAS = 90           # ⚠️ la salida domina el costo

# ── Corrida ───────────────────────────────────────────────────────────────
CONCURRENCIA        = 8            # filas simultáneas
RECICLAR_CADA       = 200          # filas por runner, para no acumular sesiones
MAX_INTENTOS        = 5            # reintentos ante 429 / 5xx
CHECKPOINT_PATH     = "checkpoint.jsonl"

# ── Salida ────────────────────────────────────────────────────────────────
SALIDA_ANCHO        = "tweets_calificados.csv"
SALIDA_TIDY         = "tweets_calificados_tidy.csv"
SALIDA_DIR_DRIVE    = None         # p.ej. "/content/drive/MyDrive/EvaluadorTweets"

# ── Visualización  (§ 8) ──────────────────────────────────────────────────
# Los períodos son intervalos [desde, hasta): límite inferior INCLUSIVO y
# superior EXCLUSIVO, para que un tuit del 11 de junio caiga en uno solo.
# El último queda abierto (hasta=None).
# El corte del 6 de julio no es arbitrario: el último partido jugado en
# territorio mexicano fueron los octavos del 5 de julio en el Estadio Azteca.
# Guadalajara cerró el 26 de junio y Monterrey también terminó en octavos.
# El torneo siguió hasta el 19 de julio FUERA de México — de ahí el nombre
# del tercer período.
# 'desde' del primero es 2026-05-31, no 2026-06-01: el corpus se recolectó por
# fecha UTC y al pasar el día a hora de México 13 tuits retroceden al 31 de mayo.
# Empezar el 1 de junio los dejaría fuera de todo período sin motivo.
PERIODOS = [
    {"nombre": "PREVIO A MUNDIAL",              "desde": "2026-05-31", "hasta": "2026-06-11"},
    {"nombre": "DURANTE MUNDIAL EN MEXICO",     "desde": "2026-06-11", "hasta": "2026-07-06"},
    {"nombre": "DESPUES DE PARTIDOS EN MEXICO", "desde": "2026-07-06", "hasta": None},
]

# 'created_at' viene en UTC y los cortes son eventos locales. Un tuit de las
# 19:00 del 10 de junio en CDMX se registra como 01:00 del 11 en UTC: sin
# convertir, cruzaría de período un día antes de tiempo. Con ~42 tuits diarios
# las seis horas mueven del orden de diez tuits en cada frontera.
ZONA_HORARIA = "America/Mexico_City"

# Niveles que significan "el tuit no trae nada de esta dimensión", NO un grado
# bajo de logro. En esta rúbrica es el nivel «0»: los niveles 1..5 son una
# escala de valencia (muy negativo → muy positivo), así que apilar el 0 junto
# a ellos lo colocaría en el extremo negativo, que es lo contrario de lo que
# significa.
#   "auto" → etiqueta «0» o puntos == 0
#   {}     → ningún nivel es de ausencia; se grafican todos
#   {"slug_del_criterio": ["0", "N/A"]} → declaración explícita por criterio
NIVELES_AUSENCIA = "auto"

# Cómo se colorean los niveles:
#   "divergente" → escala de VALENCIA (muy negativo … muy positivo). Rojo, gris
#                  neutro al centro, azul. Es la de esta rúbrica.
#   "secuencial" → escala de LOGRO (bajo … alto). Un solo matiz, claro → oscuro.
ESCALA_NIVELES = "divergente"

UMBRAL_MUESTRA_PEQUENA  = 100   # menos tuits que esto en la selección → advertir
UMBRAL_COBERTURA_DIARIA = 10    # menos calificados por día → series no interpretables
VENTANA_SUAVIZADO       = 7     # días de la media móvil (arranca APAGADA)
SALIDA_HTML             = "graficas_calificaciones.html"


def _validar_periodos(periodos) -> None:
    """Los períodos alimentan toda la § 8. Un solapamiento aquí se vuelve un
    tuit contado dos veces en las gráficas, y eso no se ve: se lee como señal."""
    if not periodos:
        raise ValueError("PERIODOS está vacío: la § 8 no tendría nada que segmentar.")

    nombres = [p.get("nombre", "") for p in periodos]
    if any(not n for n in nombres):
        raise ValueError("Todo período necesita 'nombre': es la etiqueta de las gráficas.")
    if len(set(nombres)) != len(nombres):
        raise ValueError(f"Nombres de período duplicados en PERIODOS: {nombres}")

    from datetime import date

    def _fecha(v, campo, nombre):
        if v is None:
            return None
        try:
            return date.fromisoformat(str(v))
        except ValueError as exc:
            raise ValueError(
                f"'{campo}' del período «{nombre}» no es una fecha ISO (AAAA-MM-DD): {v!r}"
            ) from exc

    anterior_desde = None
    anterior_hasta = None
    for k, p in enumerate(periodos):
        desde = _fecha(p.get("desde"), "desde", p["nombre"])
        hasta = _fecha(p.get("hasta"), "hasta", p["nombre"])
        if desde is None:
            raise ValueError(f"El período «{p['nombre']}» no declara 'desde'.")
        if hasta is not None and hasta <= desde:
            raise ValueError(
                f"El período «{p['nombre']}» termina antes de empezar: {desde} → {hasta}"
            )
        if hasta is None and k != len(periodos) - 1:
            raise ValueError(
                f"Sólo el último período puede quedar abierto, y «{p['nombre']}» "
                f"no lo es. Declara su 'hasta'."
            )
        if anterior_desde is not None and desde < anterior_desde:
            raise ValueError(
                f"Los períodos no están ordenados: «{p['nombre']}» empieza en {desde}, "
                f"antes que «{periodos[k-1]['nombre']}» ({anterior_desde}). "
                f"Decláralos en orden cronológico."
            )
        if anterior_hasta is not None and desde < anterior_hasta:
            raise ValueError(
                f"Los períodos se solapan: «{periodos[k-1]['nombre']}» llega hasta "
                f"{anterior_hasta} y «{p['nombre']}» empieza en {desde}. Con límites "
                f"[desde, hasta) el corte compartido debe ser la MISMA fecha."
            )
        anterior_desde, anterior_hasta = desde, hasta

_validar_periodos(PERIODOS)

# ── Precios USD por 1M tokens (actualizar si cambian) ─────────────────────
PRECIOS = {
    "gemini-2.5-flash":      (0.30, 2.50),
    "gemini-2.5-flash-lite": (0.10, 0.40),
    "gemini-2.5-pro":        (1.25, 10.00),
    "gemini-3.7-flash":      (0.75, 3.75),
}

# ── Umbrales de las verificaciones de calidad (§6) ────────────────────────
UMBRAL_CONCENTRACION = 0.90   # un nivel con más de esto → criterio sin poder discriminante
UMBRAL_NO_APLICABLE  = 0.50   # más de esto no aplicable → la rúbrica puede no corresponder
UMBRAL_ESTABILIDAD   = 0.95   # acuerdo test-retest mínimo esperado con temperature=0

print(f"Modelo {MODEL_ID} · thinking={'apagado' if THINKING_BUDGET == 0 else THINKING_BUDGET}"
      f" · temp={TEMPERATURE} · muestra={N_MUESTRA or 'completa'} · concurrencia={CONCURRENCIA}")
print(f"Períodos  {len(PERIODOS)} · día en {ZONA_HORARIA} · "
      f"{' | '.join(p['nombre'] for p in PERIODOS)}")

In [ ]:
#@title 3 · Autenticación y event loop
import os, sys, asyncio

# ── Event loop: Colab ya tiene uno corriendo; ADK es asíncrono ────────────
import nest_asyncio
nest_asyncio.apply()

def run_async(coro):
    """Ejecuta una coroutine desde una celda de Colab."""
    return asyncio.get_event_loop().run_until_complete(coro)

# ── Clave de API desde el gestor de secretos ──────────────────────────────
def cargar_api_key(secret_name: str = SECRET_NAME) -> str:
    try:
        from google.colab import userdata
    except ImportError:
        clave = os.environ.get(secret_name)
        if not clave:
            raise RuntimeError(
                f"Fuera de Colab: exporta {secret_name} como variable de entorno."
            )
        return clave

    try:
        clave = userdata.get(secret_name)
    except Exception as exc:
        raise RuntimeError(
            f"No se pudo leer el secreto '{secret_name}'.\n"
            f"Regístralo en el panel 🔑 (izquierda) → 'Nuevo secreto', "
            f"nómbralo '{secret_name}', pega tu clave de Google AI Studio "
            f"(https://aistudio.google.com/apikey) y activa "
            f"'Acceso del notebook'.\nDetalle: {exc}"
        ) from exc

    if not clave or not clave.strip():
        raise RuntimeError(f"El secreto '{secret_name}' está vacío.")
    return clave.strip()

_API_KEY = cargar_api_key()

# IMPORTANTE: en ADK 1.25 el cliente de google-genai se construye SIN argumentos
# (google/adk/models/google_llm.py → api_client). Sólo lee variables de entorno,
# así que pasar api_key= o vertexai= al objeto Gemini no tiene efecto.
os.environ["GOOGLE_API_KEY"] = _API_KEY
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "False"   # AI Studio, no Vertex AI

print(f"✅ Clave cargada desde el secreto '{SECRET_NAME}' "
      f"({len(_API_KEY)} caracteres, no se imprime el valor)")
print("   Backend: Gemini API (AI Studio)")

In [ ]:
#@title 4 · Sondeo: validar que el modelo exista para esta credencial
from google import genai

def validar_modelos(*model_ids) -> None:
    cliente = genai.Client()
    try:
        disponibles = {
            m.name.split("/")[-1]
            for m in cliente.models.list()
            if "generateContent" in (getattr(m, "supported_actions", None) or ["generateContent"])
        }
    except Exception as exc:
        raise RuntimeError(
            f"No se pudo listar modelos. La clave puede ser inválida o sin permisos.\n{exc}"
        ) from exc

    faltantes = [m for m in model_ids if m and m not in disponibles]
    if faltantes:
        flash = sorted(m for m in disponibles if "flash" in m or "pro" in m)
        raise RuntimeError(
            f"Modelo(s) no disponible(s) para esta credencial: {faltantes}\n"
            f"Disponibles (flash/pro): {flash}\n"
            f"Corrige MODEL_ID en la celda CONFIG."
        )
    print(f"✅ Modelos validados: {[m for m in model_ids if m]}")
    print(f"   ({len(disponibles)} modelos accesibles con esta clave)")

validar_modelos(MODEL_ID, MODEL_ID_CALIBRACION)

---
## § 2 · Rúbrica

El PDF se lee **nativamente** con Gemini (acepta `application/pdf` como entrada), sin pypdf ni OCR. El resultado se normaliza a `rubrica.json`, que es un **punto de control humano**: revísalo antes de gastar en la corrida.

Deliberadamente **no** se usa el corpus RAG de Vertex AI que emplea `Ajuste Evaluador`. La rúbrica viaja en el prompt: elimina una llamada de retrieval por evaluación, y al no haber herramientas los agentes pueden usar `output_schema` (en ADK, `tools` y `output_schema` son mutuamente excluyentes).

In [ ]:
#@title 5 · Esquema de la rúbrica  ·  agnóstico a N y al tipo de escala
from typing import List, Optional, Literal
from pydantic import BaseModel, Field

class Nivel(BaseModel):
    etiqueta: str = Field(description="Nombre del nivel tal como aparece en la rúbrica, p.ej. 'Experto', 'Logrado', '0.6 pts'")
    puntos: Optional[float] = Field(default=None, description="Valor numérico del nivel si la rúbrica lo define; null si la escala es nominal")
    descriptor: str = Field(description="Descripción completa y literal del nivel, sin resumir. Es el texto contra el que se califica.")

class Criterio(BaseModel):
    id_criterio: str = Field(description="Identificador corto tal como aparece en la rúbrica, p.ej. 'Q1', 'D3', 'C2'. Si no hay, usa un ordinal: '1', '2'.")
    nombre: str = Field(description="Nombre o título del criterio")
    consigna: str = Field(description="La pregunta o consigna de evaluación completa del criterio")
    niveles: List[Nivel] = Field(description="Niveles de logro, ordenados del más bajo al más alto")

class Rubrica(BaseModel):
    criterios: List[Criterio]

print("Esquema definido: Rubrica → Criterio[] → Nivel[]")
print("'puntos' es opcional: soporta escalas numéricas y nominales.")

In [ ]:
#@title 6 · Cargar el PDF de la rúbrica
import io
from pathlib import Path

def cargar_pdf_rubrica(path=None) -> tuple:
    """Devuelve (nombre, bytes) del PDF. Acepta ruta o subida interactiva."""
    if path:
        p = Path(path)
        if p.suffix.lower() != ".pdf":
            raise ValueError(f"Sólo se acepta PDF. Recibido: '{p.suffix}'")
        if not p.exists():
            raise FileNotFoundError(f"No existe: {p}")
        datos = p.read_bytes()
        print(f"✅ PDF cargado desde disco: {p.name} ({len(datos):,} bytes)")
        return p.name, datos

    from google.colab import files
    subidos = files.upload()
    if not subidos:
        raise RuntimeError("No se subió ningún archivo.")
    nombre = next(iter(subidos))
    if not nombre.lower().endswith(".pdf"):
        raise ValueError(
            f"Sólo se acepta PDF. Recibido: '{nombre}'. "
            "Si tu rúbrica está en Word, expórtala a PDF primero."
        )
    datos = subidos[nombre]
    print(f"✅ PDF cargado: {nombre} ({len(datos):,} bytes)")
    return nombre, datos

RUBRICA_PDF_NOMBRE, RUBRICA_PDF_BYTES = cargar_pdf_rubrica(RUBRICA_PDF_PATH)

In [ ]:
#@title 7 · Utilidad compartida: ejecutar un agente de ADK y recoger su estado
from google.adk.agents import LlmAgent, ParallelAgent
from google.adk.models.google_llm import Gemini
from google.adk.runners import InMemoryRunner
from google.genai import types
import uuid

RETRY_OPTIONS = types.HttpRetryOptions(
    attempts=MAX_INTENTOS, exp_base=7, initial_delay=1,
    http_status_codes=[429, 500, 502, 503, 504],
)

def construir_modelo(model_id: str = None) -> Gemini:
    """retry_options SÍ es un campo real de Gemini en ADK 1.25.
    api_key/vertexai/project NO lo son: se configuran por variables de entorno."""
    return Gemini(model=model_id or MODEL_ID, retry_options=RETRY_OPTIONS)

def config_generacion() -> types.GenerateContentConfig:
    """temperature y thinking_config van aquí, en LlmAgent.generate_content_config.
    No en el constructor de Gemini, que los ignora."""
    return types.GenerateContentConfig(
        temperature=TEMPERATURE,
        thinking_config=types.ThinkingConfig(thinking_budget=THINKING_BUDGET),
    )

async def ejecutar_agente(runner: InMemoryRunner, partes: list, claves: list) -> dict:
    """Corre el agente del runner con `partes` como mensaje y devuelve
    {clave_de_estado: valor} para las claves pedidas.

    Cada invocación usa una sesión nueva: evita fuga de estado entre filas,
    igual que hace Ajuste Evaluador con su session_id por job.
    """
    session_id = f"s_{uuid.uuid4().hex}"
    await runner.session_service.create_session(
        session_id=session_id, user_id="u", app_name=runner.app_name
    )
    recogido = {}
    try:
        async for evento in runner.run_async(
            user_id="u",
            session_id=session_id,
            new_message=types.Content(role="user", parts=partes),
        ):
            delta = getattr(getattr(evento, "actions", None), "state_delta", None)
            if delta:
                for k in claves:
                    if k in delta:
                        recogido[k] = delta[k]
    finally:
        try:
            await runner.session_service.delete_session(
                session_id=session_id, user_id="u", app_name=runner.app_name
            )
        except Exception:
            pass   # no todas las versiones exponen delete_session
    return recogido

print("Utilidades listas: construir_modelo, config_generacion, ejecutar_agente")

In [ ]:
#@title 8 · Parsear el PDF → rubrica.json  (+ identificadores, escala, resumen)
import json, re, unicodedata
from pathlib import Path

INSTRUCCION_PARSER = (
    "Eres un extractor de rúbricas de evaluación. Recibes el PDF de una rúbrica.\n"
    "Tu tarea es transcribir su estructura EXACTA a datos, sin interpretarla ni mejorarla.\n\n"
    "REGLAS OBLIGATORIAS:\n"
    "1. Extrae TODOS los criterios que contenga el documento. No omitas ninguno.\n"
    "2. Transcribe el descriptor de cada nivel COMPLETO y LITERAL. No resumas, no parafrasees, "
    "no acortes. Ese texto es la vara con la que se calificará después.\n"
    "3. Ordena los niveles del MÁS BAJO al MÁS ALTO de logro.\n"
    "4. Si la rúbrica asigna valores numéricos a los niveles, ponlos en 'puntos'. "
    "Si los niveles son sólo nominales (por ejemplo 'Logrado', 'En proceso', 'Inicial'), "
    "deja 'puntos' en null. NO inventes valores numéricos que la rúbrica no declara.\n"
    "5. Usa como 'id_criterio' el identificador que aparezca en el documento "
    "(Q1, D3, C2, Criterio 4...). Si no hay ninguno, usa un ordinal.\n"
    "6. NO inventes criterios ni niveles que no estén en el PDF.\n"
    "7. Conserva el idioma original del documento."
)

def _slug(texto: str, maxlen: int = 28) -> str:
    t = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode()
    t = re.sub(r"[^a-zA-Z0-9]+", "_", t).strip("_").lower()
    return (t[:maxlen].rstrip("_") or "crit")

def _asignar_slugs(rubrica: dict) -> dict:
    """Identificadores únicos, estables y aptos para nombres de columna."""
    vistos = {}
    for i, c in enumerate(rubrica["criterios"], start=1):
        base = _slug(f"{c.get('id_criterio') or i}_{c.get('nombre') or ''}")
        slug = base
        if base in vistos:
            vistos[base] += 1
            slug = f"{base}_{vistos[base]}"
            print(f"   ⚠️  colisión de identificador '{base}' → desambiguado como '{slug}'")
        else:
            vistos[base] = 1
        c["slug"] = slug
    return rubrica

def _anotar_escala(rubrica: dict) -> dict:
    for c in rubrica["criterios"]:
        puntos = [n.get("puntos") for n in c["niveles"]]
        numerica = all(p is not None for p in puntos) and len(puntos) > 0
        c["escala_numerica"] = bool(numerica)
        c["etiquetas_validas"] = [n["etiqueta"] for n in c["niveles"]]
        c["puntos_validos"] = [p for p in puntos if p is not None] if numerica else []
    return rubrica

async def _parsear_pdf(pdf_bytes: bytes) -> dict:
    agente = LlmAgent(
        name="RubricParserAgent",
        model=construir_modelo(),
        instruction=INSTRUCCION_PARSER,
        output_schema=Rubrica,
        output_key="rubrica",
        generate_content_config=config_generacion(),
        disallow_transfer_to_parent=True,
        disallow_transfer_to_peers=True,
    )
    runner = InMemoryRunner(agent=agente, app_name="rubric_parser")
    partes = [
        types.Part.from_bytes(data=pdf_bytes, mime_type="application/pdf"),
        types.Part(text="Extrae la estructura completa de esta rúbrica."),
    ]
    estado = await ejecutar_agente(runner, partes, ["rubrica"])
    if "rubrica" not in estado:
        raise RuntimeError(
            "El parseo no devolvió ninguna rúbrica. Revisa que el PDF contenga texto "
            "seleccionable (no una imagen escaneada) y vuelve a intentar."
        )
    return estado["rubrica"]

def resumen_rubrica(rubrica: dict) -> None:
    crits = rubrica["criterios"]
    n_num = sum(c["escala_numerica"] for c in crits)
    print(f"\n{'='*72}")
    print(f"RÚBRICA · {len(crits)} criterios detectados")
    print(f"Escala: {n_num} numéricos · {len(crits) - n_num} nominales")
    print(f"{'='*72}")
    for c in crits:
        tipo = "numérica" if c["escala_numerica"] else "nominal"
        esc = c["puntos_validos"] if c["escala_numerica"] else c["etiquetas_validas"]
        print(f"\n[{c['id_criterio']}] {c['nombre']}   ({c['slug']})")
        print(f"    consigna : {c['consigna'][:100]}{'…' if len(c['consigna']) > 100 else ''}")
        print(f"    niveles  : {len(c['niveles'])} · escala {tipo} · {esc}")
        cortos = [n["etiqueta"] for n in c["niveles"] if len(n.get("descriptor", "")) < 25]
        if cortos:
            print(f"    ⚠️  descriptores muy cortos en {cortos} — ¿se truncaron al parsear?")
    print(f"\n{'='*72}")
    print(f"👁  REVISA '{RUBRICA_JSON_PATH}' ANTES DE CORRER. Puedes editarlo a mano")
    print(f"    y volver a ejecutar esta celda: se recargará del archivo sin llamar al modelo.")
    print(f"{'='*72}\n")

def cargar_rubrica(forzar_reparseo: bool = False) -> dict:
    ruta = Path(RUBRICA_JSON_PATH)
    if ruta.exists() and not forzar_reparseo:
        rubrica = json.loads(ruta.read_text(encoding="utf-8"))
        print(f"♻️  Rúbrica recargada de '{ruta}' (sin llamar al modelo).")
        print(f"    Para reparsear el PDF: cargar_rubrica(forzar_reparseo=True)")
    else:
        print("Parseando el PDF con Gemini…")
        rubrica = run_async(_parsear_pdf(RUBRICA_PDF_BYTES))
        rubrica = _anotar_escala(_asignar_slugs(rubrica))
        ruta.write_text(json.dumps(rubrica, ensure_ascii=False, indent=2), encoding="utf-8")
        print(f"✅ Escrito '{ruta}'")

    if not rubrica.get("criterios"):
        raise RuntimeError("La rúbrica no contiene criterios. No se puede continuar.")

    # Reanotar siempre: permite editar el JSON a mano sin recalcular estos campos
    rubrica = _anotar_escala(_asignar_slugs(rubrica))
    resumen_rubrica(rubrica)
    return rubrica

RUBRICA = cargar_rubrica()
CRITERIOS = RUBRICA["criterios"]
N_CRITERIOS = len(CRITERIOS)

---
## § 3 · Corpus de tweets

El CSV se lee con todas las columnas como texto: los identificadores de tweet son enteros de 19 dígitos y `pandas` los convertiría a notación científica, perdiendo dígitos de forma irreversible.

Nota medida sobre este corpus: **las primeras 2,000 filas tienen el doble de payload que el promedio** (el CSV viene ordenado y la cabeza está sesgada hacia filas con traducción llena). Muestrear al azar con semilla es más representativo que tomar la cabeza.

In [ ]:
#@title 8.5 · Reparar el corpus  ·  `raw_json` es la fuente de verdad
import json, codecs, re
from pathlib import Path
import pandas as pd

# ── Mapa inverso de cp1252, completado con latin-1 en los huecos ──────────
# El daño es UTF-8 leído como cp1252. Revertirlo con .encode("cp1252") falla:
# cp1252 no define 0x81, 0x8D, 0x8F, 0x90 ni 0x9D, y el decodificador original
# dejó pasar esos bytes crudos, así que el texto mezcla caracteres cp1252
# legítimos (€ ™ œ …, todos > U+00FF) con puntos de código del rango C1.
_INV_CP1252 = {}
for _b in range(256):
    _ch = codecs.decode(bytes([_b]), "cp1252", errors="replace")
    if _ch != "�":
        _INV_CP1252.setdefault(_ch, _b)
for _b in range(256):
    _INV_CP1252.setdefault(chr(_b), _b)      # respaldo latin-1 para el rango C1


def reparar_texto(s: str) -> tuple:
    """Revierte el mojibake. Devuelve (texto, se_reparó).

    Conserva el crudo en dos casos: cuando el texto ya está sano —trae algún
    carácter que ningún mojibake de cp1252 pudo producir— y cuando los bytes
    no decodifican como UTF-8, que es lo que pasa con un texto cortado a media
    secuencia multibyte.
    """
    if not s or all(ord(c) < 128 for c in s):
        return s, False                      # ASCII puro: nada que revertir
    crudos = bytearray()
    for c in s:
        b = _INV_CP1252.get(c)
        if b is None:
            return s, False                  # carácter ajeno a cp1252: ya está sano
        crudos.append(b)
    try:
        return crudos.decode("utf-8"), True
    except UnicodeDecodeError:
        return s, False                      # irreparable: se conserva el crudo


_rep = lambda s: reparar_texto(s)[0]


# ── Lectura de la fuente, con rechazo explícito de la versión destruida ───
def _leer_fuente(path) -> pd.DataFrame:
    p = Path(path)
    if p.suffix.lower() == ".csv":
        raise RuntimeError(
            f"'{p.name}' es CSV y esta reparación necesita el .xlsx.\n"
            "El CSV de este corpus sufrió una segunda ronda de daño que sustituyó "
            "por '?' los caracteres no representables, incluido dentro de su propio "
            "'raw_json' (1,241 filas). Esa pérdida es irreversible: no hay de dónde "
            "recuperar esos bytes.\n"
            "Los dos archivos son la misma exportación fila por fila, así que el CSV "
            "bueno se REGENERA desde el .xlsx. Apunta CORPUS_FUENTE_XLSX al .xlsx."
        )
    if p.suffix.lower() not in (".xlsx", ".xlsm", ".xls"):
        raise ValueError(f"Se esperaba un Excel. Recibido: '{p.suffix or p.name}'")
    if not p.exists():
        from google.colab import files
        print(f"No se encontró '{p}'. Súbelo.")
        subidos = files.upload()
        if not subidos:
            raise RuntimeError("No se subió ningún archivo.")
        p = Path(next(iter(subidos)))

    hojas = pd.ExcelFile(p).sheet_names
    df = pd.read_excel(p, sheet_name=hojas[0], dtype=str, keep_default_na=False)
    print(f"✅ Fuente: {p.name} — hoja '{hojas[0]}' — {len(df):,} filas × {len(df.columns)} columnas")
    if "raw_json" not in df.columns:
        raise RuntimeError(
            "El archivo no trae la columna 'raw_json'. Sin ella no hay nada que "
            "reconstruir: las columnas planas están dañadas de forma irreversible."
        )
    return df


_FIRMA_SUSTITUCION = re.compile(r"[-ÿ]\?|\?[-ÿ]")
_UMBRAL_SUSTITUCION = 0.20


def _verificar_perdida_irreversible(serie) -> None:
    """Firma de la sustitución destructiva: un '?' pegado a un carácter de la
    banda 0x80-0xFF. En texto sano el '?' es puntuación y va rodeado de ASCII;
    en el archivo destruido quedó interleaved con los bytes altos del mojibake.
    Medido sobre este corpus: 0.5% de las filas en el .xlsx contra 75.6% en el
    .csv, así que un umbral de proporción separa los dos casos con holgura."""
    n = sum(1 for v in serie if _FIRMA_SUSTITUCION.search(v))
    prop = n / max(len(serie), 1)
    if prop >= _UMBRAL_SUSTITUCION:
        raise RuntimeError(
            f"{n:,} de {len(serie):,} filas ({prop:.1%}) tienen 'raw_json' con la firma "
            "de sustitución por '?' — pérdida irreversible. Este archivo no se puede "
            "reparar; usa el .xlsx original."
        )
    if n:
        print(f"   ({n} filas con '?' junto a caracteres altos — dentro de lo esperado)")


# ── Reconstrucción de una fila desde su raw_json ─────────────────────────
_PROCEDENCIA = ["base", "query_utilizado", "corpus_type",
                "window_label", "is_retweet", "is_quote"]
_SIEMPRE_VACIAS = ["origen", "source", "withheld", "tipo", "justificacion"]


def _js(v, vacio=""):
    return vacio if v is None else json.dumps(v, ensure_ascii=False, separators=(",", ":"))


def _ts(v):
    return str(v or "").replace(".000Z", "")      # el formato plano no lleva fracción ni zona


def _s(v):
    return "" if v is None else str(v)


def fila_desde_json(d: dict, plana) -> dict:
    a  = d.get("_author") or {}
    am = a.get("public_metrics") or {}
    pm = d.get("public_metrics") or {}
    p  = d.get("_place") or {}
    texto = _rep(_s(d.get("text")))
    fila = {
        "id":                   _s(d.get("id")),
        "author_id":            _s(d.get("author_id")),
        "created_at":           _ts(d.get("created_at")),
        "lang":                 _s(d.get("lang")),
        "text":                 texto,
        "text_completo":        texto,          # en este corpus son la misma cosa
        "conversation_id":      _s(d.get("conversation_id")),
        "in_reply_to_user_id":  _s(d.get("in_reply_to_user_id")),
        "possibly_sensitive":   _s(d.get("possibly_sensitive")),
        "reply_settings":       _s(d.get("reply_settings")),
        "like_count":           _s(pm.get("like_count")),
        "retweet_count":        _s(pm.get("retweet_count")),
        "reply_count":          _s(pm.get("reply_count")),
        "quote_count":          _s(pm.get("quote_count")),
        "impression_count":     _s(pm.get("impression_count")),
        "bookmark_count":       _s(pm.get("bookmark_count")),
        "referenced_tweets":    _rep(_js(d.get("referenced_tweets"), "[]")),
        "context_annotations":  _rep(_js(d.get("context_annotations"), "[]")),
        "entities":             _rep(_js(d.get("entities"))),
        "geo":                  _rep(_js(d.get("geo"))),
        "author_username":          _rep(_s(a.get("username"))),
        "author_name":              _rep(_s(a.get("name"))),
        "author_location":          _rep(_s(a.get("location"))),
        "author_description":       _rep(_s(a.get("description"))),
        "author_created_at":        _ts(a.get("created_at")),
        "author_verified":          _s(a.get("verified")),
        "author_url":               _s(a.get("url")),
        "author_profile_image_url": _s(a.get("profile_image_url")),
        "author_followers_count":   _s(am.get("followers_count")),
        "author_following_count":   _s(am.get("following_count")),
        "author_tweet_count":       _s(am.get("tweet_count")),
        "author_listed_count":      _s(am.get("listed_count")),
        "place_full_name":          _rep(_s(p.get("full_name"))),
        "place_country":            _rep(_s(p.get("country"))),
        "place_country_code":       _s(p.get("country_code")),
        "place_type":               _s(p.get("place_type")),
        "place_geo":                _js(p.get("geo")),
        "place_contained_within":   _js(p.get("contained_within")),
    }
    for c in _PROCEDENCIA:                     # no viajan en el JSON y están intactas
        fila[c] = plana.get(c, "")
    for c in _SIEMPRE_VACIAS:                  # vacías en las 2,562 filas
        fila[c] = plana.get(c, "")
    return fila


# ── Reparación completa, con reporte ─────────────────────────────────────
def reparar_corpus(fuente=None, destino=None, forzar=None) -> pd.DataFrame:
    fuente  = fuente  or CORPUS_FUENTE_XLSX
    destino = Path(destino or CORPUS_REPARADO)
    forzar  = FORZAR_REPARACION if forzar is None else forzar

    # Si el corpus ya viajó reparado (o reparado y traducido), no hace falta el
    # .xlsx de origen: es el caso de reanudar en una sesión nueva de Colab
    # subiendo sólo los artefactos.
    if not forzar:
        for ya in (CORPUS_TRADUCIDO, destino):
            if ya and Path(ya).exists():
                previo = pd.read_csv(ya, dtype=str, keep_default_na=False,
                                     low_memory=False, encoding="utf-8-sig")
                print(f"↩️  '{ya}' ya está en disco con {len(previo):,} filas — "
                      f"se omite la reparación (no se necesita el .xlsx).")
                print("   Pon FORZAR_REPARACION = True para rehacerla desde el origen.")
                return previo

    bruto = _leer_fuente(fuente)

    if destino.exists() and not forzar:
        previo = pd.read_csv(destino, dtype=str, keep_default_na=False,
                             low_memory=False, encoding="utf-8-sig")
        if len(previo) == len(bruto):
            print(f"↩️  '{destino}' ya existe con {len(previo):,} filas — se omite la reparación.")
            print("   Pon FORZAR_REPARACION = True para rehacerla.")
            return previo
        print(f"⚠️  '{destino}' existe pero tiene {len(previo):,} filas y la fuente {len(bruto):,} "
              f"— se rehace.")

    _verificar_perdida_irreversible(bruto["raw_json"])

    # Parseo previo: si algo no parsea, se aborta sin escribir nada
    docs_crudos = []
    for i, v in enumerate(bruto["raw_json"]):
        try:
            d = json.loads(v)
        except Exception as exc:
            raise RuntimeError(f"Fila {i}: 'raw_json' no parsea como JSON — {exc}")
        if not d.get("id"):
            raise RuntimeError(f"Fila {i}: 'raw_json' no declara 'id'. No se emite archivo parcial.")
        docs_crudos.append(d)

    # Reparar el propio raw_json: deja el archivo de salida uniformemente sano
    # y hace idempotente una segunda pasada sobre el archivo ya reparado.
    sanos = [reparar_texto(v)[0] for v in bruto["raw_json"]]
    docs = [json.loads(c) for c in sanos]

    filas = [fila_desde_json(docs[i], bruto.iloc[i]) for i in range(len(bruto))]
    rep = pd.DataFrame(filas)
    rep["raw_json"] = sanos
    rep = rep.reindex(columns=list(bruto.columns))    # mismo orden que la fuente

    # ── Reporte: qué se corrigió y cuánto ────────────────────────────────
    ids   = (rep["id"].astype(str) != bruto["id"].astype(str)).sum()
    convo = (rep["conversation_id"].astype(str) != bruto["conversation_id"].astype(str)).sum()
    aut   = (rep["author_id"].astype(str) != bruto["author_id"].astype(str)).sum()
    largo = sum(1 for i in range(len(rep))
                if len(rep["text_completo"].iloc[i]) > len(bruto["text_completo"].iloc[i]))
    saltos = sum(1 for i in range(len(rep))
                 if "\n" in rep["text_completo"].iloc[i]
                 and "\n" not in bruto["text_completo"].iloc[i])
    # El mojibake se cuenta sobre los valores que realmente se escribieron, es
    # decir los extraídos del raw_json crudo — no sobre las columnas planas, que
    # se descartan. Un "irreparable" aquí sí es un campo que quedó crudo en la salida.
    moji = irrep = 0
    for d in docs_crudos:
        a = d.get("_author") or {}
        for v in (_s(d.get("text")), _s(a.get("name")),
                  _s(a.get("location")), _s(a.get("description"))):
            reparado, hubo = reparar_texto(v)
            if hubo:
                moji += 1
            elif v and not all(ord(c) < 128 for c in v):
                irrep += 1

    print(f"\n{'='*70}\nREPARACIÓN\n{'='*70}")
    print(f"  identificadores corregidos   id {ids:,} · conversation_id {convo:,} · author_id {aut:,}")
    print(f"  textos destruncados          {largo:,}")
    print(f"  textos con saltos restaurados{saltos:>6,}")
    print(f"  campos con mojibake reparado {moji:,}")
    print(f"  campos irreparables          {irrep:,}  (se conserva el crudo)")

    # ── Integridad ───────────────────────────────────────────────────────
    if len(rep) != len(bruto):
        raise RuntimeError(f"Conteo de filas alterado: {len(bruto):,} → {len(rep):,}")
    unicos = rep["id"].nunique()
    largos = set(rep["id"].str.len())
    print(f"\n  filas {len(rep):,} · identificadores únicos {unicos:,} · largos {sorted(largos)}")
    if unicos != len(rep):
        print(f"  ⚠️  {len(rep) - unicos:,} identificadores duplicados tras la reparación")

    destino.parent.mkdir(parents=True, exist_ok=True)
    rep.to_csv(destino, index=False, encoding="utf-8-sig")   # el BOM es lo que salva a Excel
    print(f"\n✅ '{destino}'  ({destino.stat().st_size/1e6:.1f} MB)")
    return rep


CORPUS_REP = reparar_corpus()
CORPUS_REP[["id", "lang", "created_at", "text_completo"]].head(3)


In [ ]:
#@title 9 · Cargar el corpus de tweets  (CSV o Excel)
import pandas as pd
from pathlib import Path

def montar_drive_si_hace_falta(path: str) -> str:
    if str(path).startswith("/content/drive") and not Path("/content/drive").exists():
        from google.colab import drive
        drive.mount("/content/drive")
    return path

# dtype=str preserva los IDs de 19 dígitos intactos; la cadena vacía es el único NA
_LECTURA = dict(dtype=str, keep_default_na=False, na_values=[""])

def _leer_tabla(p: Path) -> pd.DataFrame:
    """Lee CSV o Excel. Falla con un mensaje claro si el formato no corresponde."""
    ext = p.suffix.lower()

    if ext in (".xlsx", ".xlsm", ".xls"):
        hojas = pd.ExcelFile(p).sheet_names
        print(f"   Excel detectado ({ext}) — hojas: {hojas}; se lee '{hojas[0]}'")
        return pd.read_excel(p, sheet_name=hojas[0], **_LECTURA)

    if ext in (".csv", ".tsv", ".txt"):
        sep = "\t" if ext == ".tsv" else ","
        for enc in ("utf-8", "utf-8-sig", "cp1252", "latin-1"):
            try:
                df = pd.read_csv(p, sep=sep, low_memory=False, encoding=enc, **_LECTURA)
            except UnicodeDecodeError:
                continue
            if enc != "utf-8":
                print(f"   ⚠️  el archivo no era utf-8; se leyó como {enc} "
                      f"— revisa los acentos antes de calificar")
            return df
        raise RuntimeError(f"No se pudo decodificar '{p.name}' con utf-8, cp1252 ni latin-1.")

    raise ValueError(
        f"Formato no soportado: '{p.suffix or p.name}'. "
        "Se acepta .csv, .tsv, .txt, .xlsx o .xls. "
        "Si subiste un .xlsx y ves este mensaje, el archivo perdió su extensión al subirlo."
    )

def cargar_csv(path=CSV_PATH) -> pd.DataFrame:
    p = Path(montar_drive_si_hace_falta(path))
    if not p.exists():
        from google.colab import files
        print(f"No se encontró '{p}'. Súbelo (38 MB tarda; considera usar Drive).")
        subidos = files.upload()
        if not subidos:
            raise RuntimeError("No se subió ningún archivo.")
        p = Path(next(iter(subidos)))
        print(f"   ℹ️  se usará el archivo subido '{p.name}', no CSV_PATH")

    df = _leer_tabla(p)
    print(f"✅ Corpus cargado: {p.name} — {len(df):,} filas × {len(df.columns)} columnas")

    faltantes = [c for c in (COL_ID, COL_TEXTO) if c not in df.columns]
    if faltantes:
        raise RuntimeError(
            f"Columnas requeridas ausentes: {faltantes}. "
            f"Columnas disponibles: {list(df.columns)}. "
            f"Ajusta COL_ID / COL_TEXTO en la celda CONFIG."
        )
    for opcional in (COL_TRADUCCION, COL_LANG):
        if opcional and opcional not in df.columns:
            print(f"   ⚠️  columna opcional '{opcional}' ausente — se ignora")

    dup = df[COL_ID].duplicated().sum()
    if dup:
        print(f"   ⚠️  {dup:,} identificadores duplicados en '{COL_ID}' "
              f"— el merge final los reportará")
    return df

DF = cargar_csv()
DF.head(2)


In [ ]:
#@title 9.5 · Traducción  ·  esquema e instrucción  (literal, no adaptada)
from typing import Optional
from pydantic import BaseModel, Field


class Traduccion(BaseModel):
    idioma_detectado: str = Field(
        description="El idioma del tweet tal como TÚ lo detectas, en español y en una palabra "
                    "(p.ej. 'inglés', 'japonés', 'español'). Si el texto no tiene contenido "
                    "lingüístico —sólo enlaces, menciones o emojis— responde 'sin texto'."
    )
    ya_esta_en_espanol: bool = Field(
        description="True si el texto YA está escrito en español y no requiere traducción. "
                    "No te fíes de ninguna etiqueta externa: decide por el texto que ves."
    )
    traduccion: str = Field(
        description="La traducción al español. Si ya_esta_en_espanol es True, repite el texto "
                    "original sin cambios."
    )


# El registro del original ES el dato: esta rúbrica mide percepción de un destino.
# Suavizar una grosería o pulir una falta de ortografía destruye justo la señal.
INSTRUCCION_TRADUCCION = (
    "Eres un traductor literal al español. Traduces tweets, uno a la vez.\n\n"
    "REGLAS OBLIGATORIAS:\n"
    "1. LITERAL, NO ADAPTADA. Traduce lo que dice, no lo que debería decir. No resumas, "
    "no expandas, no expliques, no completes lo que falte.\n"
    "2. CONSERVA EL REGISTRO. Si el original es vulgar, ofensivo, irónico o está mal escrito, "
    "la traducción debe serlo igual. NO censures, NO suavices, NO corrijas la ortografía. "
    "El tono del original es parte del dato que se va a analizar.\n"
    "3. NO TRADUZCAS: las menciones (@usuario), las etiquetas (#hashtag), las URL ni los "
    "nombres propios. Van idénticos, carácter por carácter.\n"
    "4. CONSERVA los emojis y los saltos de línea en las posiciones equivalentes.\n"
    "5. SI YA ESTÁ EN ESPAÑOL, marca ya_esta_en_espanol=true y repite el texto sin cambios.\n"
    "6. SI NO HAY NADA QUE TRADUCIR —sólo un enlace, una mención o emojis— devuelve el "
    "contenido tal cual y marca el idioma como 'sin texto'.\n"
    "7. No añadas comentarios, notas del traductor ni comillas envolventes. Sólo la traducción.\n\n"
    "Devuelve tu respuesta conforme al esquema estructurado solicitado."
)

print("Esquema definido: Traduccion(idioma_detectado, ya_esta_en_espanol, traduccion)")
print(f"Modelo de traducción: {MODEL_ID_TRADUCCION}")
print("La decisión de 'ya está en español' la toma el modelo, no la columna 'lang':")
print("   en este corpus 'lang' declara 64 'qme', 21 'und' y 8 'zxx', que no son idiomas.")


In [ ]:
#@title 9.6 · Traducir el corpus completo  ·  por fila, reanudable
import json, asyncio, random
from pathlib import Path
from google import genai
from google.genai import types

_CLIENTE_TRAD = genai.Client()

_CONFIG_TRAD = types.GenerateContentConfig(
    temperature=0.0,
    response_mime_type="application/json",
    response_schema=Traduccion,
    system_instruction=INSTRUCCION_TRADUCCION,
    thinking_config=types.ThinkingConfig(thinking_budget=0),
)


def _es_transitorio(exc) -> bool:
    """429 y 5xx se reintentan; un 400 por esquema o un 403 por permisos, no."""
    codigo = getattr(exc, "code", None) or getattr(exc, "status_code", None)
    if isinstance(codigo, int):
        return codigo == 429 or 500 <= codigo < 600
    t = str(exc)
    return any(s in t for s in ("429", "500", "502", "503", "504",
                                "RESOURCE_EXHAUSTED", "UNAVAILABLE", "DEADLINE"))


def _llamar_sync(texto: str):
    return _CLIENTE_TRAD.models.generate_content(
        model=MODEL_ID_TRADUCCION,
        contents=f"<tweet>\n{texto}\n</tweet>",
        config=_CONFIG_TRAD,
    )


async def _traducir_uno(texto: str, intentos: int = None) -> dict:
    """Usa el cliente SÍNCRONO dentro de un hilo, no el asíncrono.

    El cliente .aio de google-genai levanta 'Timeout should be used inside a
    task' bajo nest_asyncio, que este notebook aplica para poder correr
    corrutinas en Colab: aiohttp usa ceil_timeout, que exige una tarea real y
    no sobrevive al bucle reentrante que nest_asyncio instala.
    asyncio.to_thread ejecuta el cliente síncrono en un hilo aparte, con su
    propio contexto, y conserva el semáforo y la concurrencia.
    Nota: la misma incompatibilidad afecta a ADK cuando se corre fuera de
    Colab; dentro de Colab el entorno sí la tolera.
    """
    intentos = intentos or MAX_INTENTOS
    ultimo = None
    for n in range(intentos):
        try:
            r = await asyncio.to_thread(_llamar_sync, texto)
            return json.loads(r.text)
        except Exception as exc:
            ultimo = exc
            if not _es_transitorio(exc) or n == intentos - 1:
                break
            # espera exponencial con ruido, para no sincronizar los reintentos
            await asyncio.sleep(min(2 ** n + random.random(), 30))
    raise ultimo


# ── Checkpoint propio: no toca checkpoint.jsonl, que indexa por (tweet, criterio) ──
def _traducciones_hechas(path=None) -> dict:
    p = Path(path or CKPT_TRADUCCION)
    if not p.exists():
        return {}
    hechas = {}
    with p.open(encoding="utf-8") as fh:
        for linea in fh:
            linea = linea.strip()
            if not linea:
                continue
            try:
                r = json.loads(linea)
                hechas[str(r["tweet_id"])] = r
            except Exception:
                continue          # línea truncada por una desconexión: se ignora
    return hechas


def _anexar_traduccion(registro: dict, path=None) -> None:
    p = Path(path or CKPT_TRADUCCION)
    if p.exists() and p.stat().st_size:
        with p.open("rb") as prev:            # si una corrida murió a media línea,
            prev.seek(-1, 2)                  # el archivo no termina en \n y el
            cortada = prev.read(1) != b"\n"   # registro nuevo se concatenaría
        if cortada:
            with p.open("a", encoding="utf-8") as fh:
                fh.write("\n")
    with p.open("a", encoding="utf-8") as fh:
        fh.write(json.dumps(registro, ensure_ascii=False) + "\n")
        fh.flush()


def _estimar_costo_traduccion(n: int) -> float:
    if MODEL_ID_TRADUCCION not in PRECIOS:
        return 0.0
    p_in, p_out = PRECIOS[MODEL_ID_TRADUCCION]
    tok_in = len(INSTRUCCION_TRADUCCION) // 4 + 110    # instrucción + tweet medio
    tok_out = 130                                      # traducción + envoltura del esquema
    return n * (tok_in / 1e6 * p_in + tok_out / 1e6 * p_out)


async def traducir_corpus(df, path_ckpt=None) -> tuple:
    if COL_TRADUCCION not in df.columns:
        df[COL_TRADUCCION] = ""
        print(f"   columna '{COL_TRADUCCION}' creada")

    col = df[COL_TRADUCCION].fillna("").astype(str).str.strip()
    con_texto = df[COL_TEXTO].fillna("").astype(str).str.strip() != ""
    if RETRADUCIR_EXISTENTES:
        objetivo = df.index[con_texto]
        print(f"   RETRADUCIR_EXISTENTES=True: se reescriben todas las traducciones")
    else:
        objetivo = df.index[con_texto & (col == "")]

    hechas = _traducciones_hechas(path_ckpt)
    # Sólo las OK cuentan como hechas: una fila que falló por un 429 pasajero
    # debe reintentarse en la siguiente corrida, no quedar envenenada para siempre.
    ok = {k for k, v in hechas.items() if v.get("estado") == "OK"}
    pendientes = [i for i in objetivo if str(df.at[i, COL_ID]) not in ok]
    reintentos = sum(1 for i in pendientes if str(df.at[i, COL_ID]) in hechas)

    print(f"\nfilas con texto           {con_texto.sum():,}")
    print(f"ya traducidas en el corpus{len(df) - len(objetivo) - (~con_texto).sum():>7,}")
    print(f"recuperadas del checkpoint{len(objetivo) - len(pendientes):>7,}")
    print(f"pendientes de traducir    {len(pendientes):>7,}")
    if reintentos:
        print(f"   de ellas, {reintentos:,} fallaron en una corrida previa y se reintentan")
    if pendientes:
        print(f"costo estimado            ${_estimar_costo_traduccion(len(pendientes)):.2f} USD "
              f"· modelo {MODEL_ID_TRADUCCION}")

    fallidas = []
    if pendientes:
        sem = asyncio.Semaphore(CONCURRENCIA_TRAD)
        hecho = 0

        async def una(i):
            nonlocal hecho
            tid = str(df.at[i, COL_ID])
            async with sem:
                try:
                    r = await _traducir_uno(str(df.at[i, COL_TEXTO]))
                    reg = {"tweet_id": tid,
                           "idioma_detectado": r.get("idioma_detectado", ""),
                           "ya_esta_en_espanol": bool(r.get("ya_esta_en_espanol")),
                           "traduccion": (r.get("traduccion") or "").strip(),
                           "estado": "OK"}
                except Exception as exc:
                    reg = {"tweet_id": tid, "idioma_detectado": "", "ya_esta_en_espanol": False,
                           "traduccion": "", "estado": "FALLIDA", "detalle": str(exc)[:200]}
                    fallidas.append(tid)
                _anexar_traduccion(reg, path_ckpt)
                hechas[tid] = reg
                hecho += 1
                if hecho % 100 == 0 or hecho == len(pendientes):
                    print(f"   {hecho:,}/{len(pendientes):,} "
                          f"({hecho/len(pendientes):.0%}) · fallidas {len(fallidas):,}")

        await asyncio.gather(*(una(i) for i in pendientes))

    # ── Volcado al DataFrame ─────────────────────────────────────────────
    # El español se copia VERBATIM desde el corpus, no desde lo que devolvió el
    # modelo: así la copia es exacta por construcción y no depende de que no
    # haya alterado nada. Es la convención que ya traía el corpus anterior.
    n_trad = n_copia = 0
    for i in objetivo:
        reg = hechas.get(str(df.at[i, COL_ID]))
        if not reg or reg.get("estado") != "OK":
            continue
        if reg.get("ya_esta_en_espanol"):
            df.at[i, COL_TRADUCCION] = str(df.at[i, COL_TEXTO])
            n_copia += 1
        else:
            df.at[i, COL_TRADUCCION] = reg["traduccion"]
            n_trad += 1

    llenas = (df[COL_TRADUCCION].fillna("").astype(str).str.strip() != "").sum()
    print(f"\n{'='*66}\nTRADUCCIÓN\n{'='*66}")
    print(f"  traducidas por el modelo   {n_trad:,}")
    print(f"  copiadas verbatim (español){n_copia:>7,}")
    print(f"  fallidas                   {len(fallidas):,}")
    print(f"  columna '{COL_TRADUCCION}' llena en {llenas:,}/{len(df):,} filas")
    return df, fallidas


def _verificar_tokens(df, n: int = 60) -> None:
    """Los @handles, #hashtags y URL deben sobrevivir idénticos. Comprobación
    determinista y barata sobre una muestra."""
    import re
    pat = re.compile(r"https?://\S+|[@#]\w+")
    m = df[(df[COL_TRADUCCION].fillna("").astype(str).str.strip() != "")]
    if m.empty:
        return
    m = m.sample(n=min(n, len(m)), random_state=SEED)
    malas = []
    for _, f in m.iterrows():
        o = set(pat.findall(str(f[COL_TEXTO])))
        t = set(pat.findall(str(f[COL_TRADUCCION])))
        if o - t:
            malas.append((str(f[COL_ID]), sorted(o - t)[:3]))
    print(f"\n  tokens preservados: {len(m)-len(malas)}/{len(m)} de la muestra")
    for tid, faltan in malas[:5]:
        print(f"     ⚠️  {tid}: faltan {faltan}")



_ALIAS_IDIOMA = {
    "en": "inglés", "ja": "japonés", "es": "español", "pt": "portugués",
    "fr": "francés", "de": "alemán", "it": "italiano", "ko": "coreano",
    "nl": "neerlandés", "tl": "tagalo", "ht": "criollo haitiano", "th": "tailandés",
}
_SIN_IDIOMA = {"qme", "und", "zxx", "qam", "qht", "qst", "art", ""}


def contrastar_idioma(df, path_ckpt=None) -> None:
    """La columna 'lang' viene de la API y miente: declara 64 'qme', 21 'und' y
    8 'zxx', que no son idiomas. Contrastarla contra lo que detectó el modelo
    dice cuánto, y es gratis porque el dato ya viaja en el esquema de respuesta."""
    hechas = _traducciones_hechas(path_ckpt)
    if not hechas:
        return
    filas = []
    for _, f in df.iterrows():
        r = hechas.get(str(f[COL_ID]))
        if not r or r.get("estado") != "OK":
            continue
        lang = str(f.get(COL_LANG, "") or "").strip().lower()
        det = str(r.get("idioma_detectado", "")).strip().lower()
        filas.append((lang, det))
    if not filas:
        return

    reales = [(l, d) for l, d in filas if l not in _SIN_IDIOMA]
    de_acuerdo = sum(1 for l, d in reales if _ALIAS_IDIOMA.get(l, l) == d)
    print(f"\n{'='*66}\nCONTRASTE 'lang' vs idioma detectado\n{'='*66}")
    if reales:
        print(f"  filas con 'lang' que sí es un idioma: {len(reales):,}")
        print(f"  coinciden: {de_acuerdo:,} ({de_acuerdo/len(reales):.1%}) · "
              f"desacuerdo: {len(reales)-de_acuerdo:,}")

    sin = [(l, d) for l, d in filas if l in _SIN_IDIOMA]
    if sin:
        from collections import Counter
        print(f"\n  filas cuyo 'lang' NO es un idioma: {len(sin):,}")
        for cod in sorted({l for l, _ in sin}):
            c = Counter(d for l, d in sin if l == cod)
            top = ", ".join(f"{k}:{v}" for k, v in c.most_common(4))
            print(f"     '{cod or '(vacío)'}' ({sum(c.values())}) → {top}")

    from collections import Counter
    desac = Counter((l, d) for l, d in reales if _ALIAS_IDIOMA.get(l, l) != d)
    if desac:
        print(f"\n  desacuerdos más frecuentes:")
        for (l, d), n in desac.most_common(5):
            print(f"     lang='{l}' pero el modelo detectó '{d}'  ({n})")


_dest = Path(CORPUS_TRADUCIDO)
if _dest.exists() and not RETRADUCIR_EXISTENTES and CSV_PATH == CORPUS_TRADUCIDO:
    print(f"↩️  '{_dest}' ya es el corpus cargado y no hay retraducción pedida.")
    DF, TRAD_FALLIDAS = DF, []
else:
    DF, TRAD_FALLIDAS = run_async(traducir_corpus(DF))
    DF.to_csv(_dest, index=False, encoding="utf-8-sig")     # el BOM salva a Excel
    print(f"\n✅ '{_dest}'  ({_dest.stat().st_size/1e6:.1f} MB)")

_verificar_tokens(DF)
contrastar_idioma(DF)
DF[[COL_ID, COL_LANG, COL_TEXTO, COL_TRADUCCION]].head(3)


In [ ]:
#@title 10 · Seleccionar el subconjunto a evaluar
def seleccionar_subconjunto(df: pd.DataFrame) -> pd.DataFrame:
    sub = df.copy()
    inicial = len(sub)
    trazas = []

    def _es_true(serie):
        return serie.fillna("").astype(str).str.strip().str.upper().isin(["TRUE", "1", "SI", "YES"])

    if EXCLUIR_RETWEETS and "is_retweet" in sub.columns:
        n = _es_true(sub["is_retweet"]).sum()
        sub = sub[~_es_true(sub["is_retweet"])]
        trazas.append(f"retweets excluidos: {n:,}")
    if EXCLUIR_QUOTES and "is_quote" in sub.columns:
        n = _es_true(sub["is_quote"]).sum()
        sub = sub[~_es_true(sub["is_quote"])]
        trazas.append(f"quotes excluidos: {n:,}")
    if FILTRO_LANG and COL_LANG in sub.columns:
        antes = len(sub)
        sub = sub[sub[COL_LANG].isin(FILTRO_LANG)]
        trazas.append(f"filtro lang {FILTRO_LANG}: {antes - len(sub):,} fuera")
    if FILTRO_TIPO_QUERY and "TIPO_QUERY" in sub.columns:
        antes = len(sub)
        sub = sub[sub["TIPO_QUERY"].isin(FILTRO_TIPO_QUERY)]
        trazas.append(f"filtro TIPO_QUERY {FILTRO_TIPO_QUERY}: {antes - len(sub):,} fuera")

    # Descartar filas sin texto: no hay nada que calificar
    vacias = sub[COL_TEXTO].fillna("").str.strip().eq("")
    if vacias.any():
        trazas.append(f"sin texto: {vacias.sum():,} fuera")
        sub = sub[~vacias]

    if N_MUESTRA is not None and N_MUESTRA < len(sub):
        sub = sub.sample(n=N_MUESTRA, random_state=SEED)
        trazas.append(f"muestra aleatoria n={N_MUESTRA} (seed={SEED})")

    print(f"Subconjunto: {inicial:,} → {len(sub):,} filas")
    for t in trazas:
        print(f"   · {t}")
    print(f"\n{len(sub):,} filas × {N_CRITERIOS} criterios = "
          f"{len(sub) * N_CRITERIOS:,} llamadas al modelo")
    return sub.reset_index(drop=True)

SUB = seleccionar_subconjunto(DF)

In [ ]:
#@title 11 · Construir el payload de evaluación  (multilingüe + contexto incompleto)
def _txt(fila, col) -> str:
    """Lectura segura de una celda. CUIDADO: NaN de pandas es TRUTHY, así que
    `if fila.get(col):` da True en celdas vacías. Sin esto, con el CSV real las
    13,412 filas de 'in_reply_to_user_id' vacío se marcarían todas como reply,
    y cada fila sin traducción recibiría el literal 'nan' como traducción."""
    if not col:
        return ""
    v = fila.get(col)
    if v is None or (isinstance(v, float) and v != v) or pd.isna(v):
        return ""
    s = str(v).strip()
    return "" if s.lower() in {"nan", "none", "<na>"} else s

def _es_true(v) -> bool:
    if v is None or (isinstance(v, float) and v != v):
        return False
    return str(v).strip().upper() in {"TRUE", "1", "SI", "YES"}

def detectar_contexto_incompleto(fila) -> list:
    """Replies, retweets y quotes: su texto no es autosuficiente."""
    marcas = []
    if _txt(fila, "in_reply_to_user_id"):
        marcas.append("reply")
    elif _txt(fila, COL_TEXTO).lstrip().startswith("@"):
        marcas.append("reply")          # heurística: arranca con mención
    if _es_true(fila.get("is_retweet")):
        marcas.append("retweet")
    if _es_true(fila.get("is_quote")):
        marcas.append("quote")
    return marcas

_AVISOS = {
    "reply":   "Este tweet es una RESPUESTA a otro tweet que NO está disponible. "
               "Califica sólo con el texto presente y di en tu justificación que el contexto previo falta.",
    "retweet": "Este tweet es un RETWEET: el texto no fue escrito por el autor de la fila.",
    "quote":   "Este tweet CITA otro tweet que NO está disponible.",
}

def construir_payload(fila) -> tuple:
    """Devuelve (texto_payload, marcas_de_contexto)."""
    partes = []
    marcas = detectar_contexto_incompleto(fila)

    if marcas:
        avisos = " ".join(_AVISOS[m] for m in marcas if m in _AVISOS)
        partes.append(f"[ADVERTENCIA DE CONTEXTO INCOMPLETO] {avisos}")

    lang = _txt(fila, COL_LANG) or "desconocido"
    partes.append(f"<tweet idioma=\"{lang}\">\n{_txt(fila, COL_TEXTO)}\n</tweet>")

    trad = _txt(fila, COL_TRADUCCION)
    if trad:
        partes.append(
            f"<traduccion_al_espanol_de_apoyo>\n{trad}\n"
            f"</traduccion_al_espanol_de_apoyo>"
        )
    # Sin traducción no se invoca ninguna: el modelo es multilingüe.
    return "\n\n".join(partes), marcas

SUB["_payload"], SUB["_marcas_contexto"] = zip(*[construir_payload(f) for _, f in SUB.iterrows()])
SUB["contexto_incompleto"] = SUB["_marcas_contexto"].apply(lambda m: ";".join(m) if m else "")

n_inc = (SUB["contexto_incompleto"] != "").sum()
print(f"Payloads construidos: {len(SUB):,}")
print(f"   con contexto incompleto: {n_inc:,} ({n_inc/max(len(SUB),1):.1%})")
print(f"   con traducción de apoyo: {sum('traduccion_al_espanol' in p for p in SUB['_payload']):,}")
print("\n── Ejemplo de payload ──")
print(SUB["_payload"].iloc[0][:700])

---
## § 4 · Agentes de calificación

Un agente por criterio, construido en un bucle sobre `rubrica.json`. **El número de criterios no aparece en el código.**

Cada agente ve **sólo su criterio** — así hereda el aislamiento de prompts de `Ajuste Evaluador`, que evita que un criterio contamine el juicio de otro. Los N agentes se agrupan en un `ParallelAgent`, de modo que los criterios de un mismo tweet se evalúan concurrentemente.

Dos diferencias deliberadas frente a `Ajuste Evaluador`:

- **`output_schema` de Pydantic** en vez de una expresión regular sobre `**Puntuación: X.X**`. A miles de filas, el texto libre no es parseable de forma confiable.
- **El modelo devuelve la *etiqueta* del nivel, no el puntaje.** El puntaje se deriva después de forma determinista consultando `rubrica.json`. Así es imposible que el modelo invente un 0.35 en una escala que sólo admite 0.0, 0.2, 0.4, 0.6, 0.8 y 1.0.

In [ ]:
#@title 12 · Esquema de una calificación
class Calificacion(BaseModel):
    aplicable: bool = Field(
        description="True si el contenido del tweet tiene relación con lo que este criterio evalúa. "
                    "False si el tweet es ajeno al criterio. OJO: 'aplicable=False' NO es lo mismo "
                    "que el nivel más bajo; un tweet fuera de alcance no es un tweet de bajo logro."
    )
    nivel: Optional[str] = Field(
        default=None,
        description="La etiqueta EXACTA del nivel de logro que refleja el tweet, copiada literalmente "
                    "de la lista de niveles válidos. null si aplicable=False."
    )
    justificacion: str = Field(
        description="Por qué se asigna ese nivel: cita el contenido concreto del tweet y relaciónalo "
                    "con el descriptor del nivel elegido. En ESPAÑOL. Si aplicable=False, explica "
                    "por qué el criterio queda fuera de alcance para este tweet."
    )

print("Esquema definido: Calificacion(aplicable, nivel, justificacion)")
print("El puntaje NO lo genera el modelo: se deriva de rubrica.json.")

In [ ]:
#@title 13 · Plantilla de instrucción por criterio
def instruccion_para(criterio: dict) -> str:
    """Criterio PRIMERO, tweet AL FINAL: deja el prefijo idéntico entre llamadas
    del mismo criterio, elegible para el caché implícito de Gemini."""
    niveles = "\n".join(
        f"  · «{n['etiqueta']}»"
        + (f" ({n['puntos']} pts)" if n.get("puntos") is not None else "")
        + f": {n['descriptor']}"
        for n in criterio["niveles"]
    )
    etiquetas = " | ".join(f"«{e}»" for e in criterio["etiquetas_validas"])

    return (
        f"Eres evaluador experto del criterio [{criterio['id_criterio']}] "
        f"«{criterio['nombre']}» de una rúbrica de evaluación.\n"
        f"Evalúas UN tweet a la vez, contra ESTE criterio y ningún otro.\n\n"
        f"CONSIGNA DEL CRITERIO:\n{criterio['consigna']}\n\n"
        f"NIVELES DE LOGRO (del más bajo al más alto):\n{niveles}\n\n"
        f"ESCALA VÁLIDA — el campo 'nivel' DEBE ser exactamente una de estas etiquetas, "
        f"copiada literalmente:\n{etiquetas}\n\n"
        f"REGLAS OBLIGATORIAS:\n"
        f"1. APLICABILIDAD PRIMERO. Si el contenido del tweet es ajeno a lo que este criterio "
        f"evalúa, responde aplicable=false, nivel=null, y explica en la justificación por qué "
        f"queda fuera de alcance. NO asignes el nivel más bajo a un tweet que simplemente no "
        f"trata el tema: 'fuera de alcance' y 'bajo logro' son cosas distintas.\n"
        f"2. NO INVENTES. No infieras, no supongas ni des crédito por contenido que no está "
        f"presente en el texto del tweet. Si algo que el criterio requiere no aparece, dilo.\n"
        f"3. CONTEXTO INCOMPLETO. Si el tweet viene marcado con una advertencia de contexto "
        f"incompleto, califica ÚNICAMENTE con el texto disponible, reconoce la limitación en tu "
        f"justificación y NO reconstruyas la conversación faltante.\n"
        f"4. ESCALA CERRADA. El nivel debe ser una de las etiquetas listadas arriba, textual. "
        f"No inventes niveles intermedios ni reformules las etiquetas.\n"
        f"5. IDIOMA. El tweet puede estar en cualquier idioma; evalúalo en su idioma original. "
        f"Tu justificación debe estar SIEMPRE en ESPAÑOL.\n"
        f"6. EXTENSIÓN. La justificación debe tener como máximo {JUSTIF_MAX_PALABRAS} palabras. "
        f"Sé específico y concreto, no genérico. No uses frases de relleno como 'Claro', "
        f"'Entiendo' o 'Voy a evaluar': empieza directo por la evaluación.\n\n"
        f"Devuelve tu respuesta conforme al esquema estructurado solicitado."
    )

print("── Ejemplo de instrucción generada ──")
print(instruccion_para(CRITERIOS[0])[:1100], "…")

In [ ]:
#@title 14 · Construir los N agentes y el ParallelAgent
def clave_estado(criterio: dict) -> str:
    return f"cal_{criterio['slug']}"

def construir_equipo(model_id: str = None) -> tuple:
    """Un agente por criterio de la rúbrica. N nunca aparece fijo en el código."""
    agentes = []
    for c in CRITERIOS:                      # ← el bucle es todo el agnosticismo
        agentes.append(LlmAgent(
            name=f"Eval_{c['slug']}",
            model=construir_modelo(model_id),
            instruction=instruccion_para(c),
            output_schema=Calificacion,
            output_key=clave_estado(c),
            generate_content_config=config_generacion(),
            disallow_transfer_to_parent=True,
            disallow_transfer_to_peers=True,
        ))
    equipo = ParallelAgent(name="EquipoCriteriosParalelo", sub_agents=agentes)
    return equipo, [clave_estado(c) for c in CRITERIOS]

EQUIPO, CLAVES = construir_equipo()

print(f"✅ {len(EQUIPO.sub_agents)} agentes construidos en ParallelAgent "
      f"(N derivado de la rúbrica, no del código)")
for a, c in zip(EQUIPO.sub_agents, CRITERIOS):
    print(f"   {a.name:<34} → estado '{clave_estado(c)}'")

# Verificación de aislamiento: la instrucción de un agente no debe mencionar
# los descriptores de los otros criterios.
if N_CRITERIOS > 1:
    otro = CRITERIOS[1]["niveles"][0]["descriptor"][:60]
    assert otro not in EQUIPO.sub_agents[0].instruction, \
        "Fuga de aislamiento: un agente ve el descriptor de otro criterio"
    print("\n✅ Aislamiento verificado: cada agente ve sólo su criterio")

In [ ]:
#@title 15 · Validar el nivel y derivar el puntaje de la rúbrica
_MAPA_PUNTOS = {
    c["slug"]: {n["etiqueta"]: n.get("puntos") for n in c["niveles"]}
    for c in CRITERIOS
}
_ETIQUETAS = {c["slug"]: set(c["etiquetas_validas"]) for c in CRITERIOS}
_NUMERICA  = {c["slug"]: c["escala_numerica"] for c in CRITERIOS}

def _normalizar(s: str) -> str:
    return re.sub(r"\s+", " ", str(s or "")).strip().strip("«»\"'").lower()

def validar_calificacion(slug: str, cruda: dict) -> dict:
    """Devuelve un registro normalizado. estado ∈ {OK, NO_APLICABLE, FUERA_DE_ESCALA}"""
    reg = {
        "slug": slug,
        "aplicable": bool(cruda.get("aplicable")),
        "nivel": None,
        "puntaje": None,
        "justificacion": (cruda.get("justificacion") or "").strip(),
        "estado": "OK",
        "detalle": "",
    }
    if not reg["aplicable"]:
        reg["estado"] = "NO_APLICABLE"
        return reg

    nivel_crudo = cruda.get("nivel")
    validas = _ETIQUETAS[slug]
    # coincidencia exacta, y si no, tolerante a espacios/comillas/mayúsculas
    if nivel_crudo in validas:
        nivel = nivel_crudo
    else:
        candidatos = [e for e in validas if _normalizar(e) == _normalizar(nivel_crudo)]
        nivel = candidatos[0] if candidatos else None

    if nivel is None:
        reg["estado"] = "FUERA_DE_ESCALA"
        reg["detalle"] = f"nivel devuelto {nivel_crudo!r} no está en la escala del criterio"
        return reg

    reg["nivel"] = nivel
    if _NUMERICA[slug]:
        reg["puntaje"] = _MAPA_PUNTOS[slug].get(nivel)   # ← determinista, de la rúbrica
    return reg

print("Validador listo. El puntaje se deriva de rubrica.json, nunca del modelo.")
print(f"Escalas: {sum(_NUMERICA.values())} numéricas, {N_CRITERIOS - sum(_NUMERICA.values())} nominales")

In [ ]:
#@title 16 · Prueba de humo sobre un tweet  (1 fila × N criterios)
async def evaluar_payload(runner, payload: str, claves: list) -> dict:
    partes = [types.Part(text=payload),
              types.Part(text="Evalúa el tweet anterior según tu criterio.")]
    return await ejecutar_agente(runner, partes, claves)

def prueba_de_humo(idx: int = 0):
    fila = SUB.iloc[idx]
    runner = InMemoryRunner(agent=EQUIPO, app_name="smoke")
    estado = run_async(evaluar_payload(runner, fila["_payload"], CLAVES))

    print(f"TWEET [{fila[COL_ID]}] lang={fila.get(COL_LANG)} "
          f"contexto={fila['contexto_incompleto'] or 'completo'}")
    print(f"  {str(fila[COL_TEXTO])[:220]}\n")
    print(f"Resultados recibidos: {len(estado)}/{N_CRITERIOS}\n")

    for c in CRITERIOS:
        k = clave_estado(c)
        if k not in estado:
            print(f"[{c['id_criterio']}] ❌ sin resultado")
            continue
        reg = validar_calificacion(c["slug"], estado[k])
        marca = {"OK": "✅", "NO_APLICABLE": "➖", "FUERA_DE_ESCALA": "⚠️"}[reg["estado"]]
        pts = f" ({reg['puntaje']} pts)" if reg["puntaje"] is not None else ""
        print(f"{marca} [{c['id_criterio']}] {c['nombre'][:44]}")
        print(f"     nivel: {reg['nivel'] or '—'}{pts}   estado: {reg['estado']}")
        print(f"     just : {reg['justificacion'][:200]}")
        if reg["detalle"]:
            print(f"     ⚠️  {reg['detalle']}")
        print()
    return estado

_ = prueba_de_humo(0)

---
## § 5 · Corrida

Dos ejes de paralelismo:

```
pool asyncio (Semaphore)                    ← throughput sobre filas
   ├── fila 1 ──▶ ParallelAgent ──┬── Agente C1   ← aislamiento por criterio
   │                              ├── Agente C2      (heredado de Ajuste Evaluador)
   │                              └── Agente CN
   ├── fila 2 ──▶ ParallelAgent ──┬── …
   └── fila k ──▶ …
```

Cada fila usa una **sesión nueva** (evita fuga de estado, como hace `Ajuste Evaluador` con su `session_id` por job) y el runner se **recicla cada `RECICLAR_CADA` filas** para no acumular miles de sesiones en memoria.

El checkpoint es **JSONL por anexado**, una línea por `(tweet, criterio)`, escrita en cuanto se obtiene. Colab se desconecta: reejecutar la celda continúa donde quedó.

In [ ]:
#@title 17 · Estimar el costo y confirmar antes de gastar
def estimar_costo(sub: pd.DataFrame, model_id: str = None, muestra_conteo: int = 15) -> dict:
    model_id = model_id or MODEL_ID
    if model_id not in PRECIOS:
        print(f"⚠️  Sin precio para '{model_id}'. Añádelo a PRECIOS en CONFIG.")
        return {}
    p_in, p_out = PRECIOS[model_id]
    cliente = genai.Client()

    # Tokens de entrada reales: instrucción del criterio + payload, medidos con la API
    filas_m = sub.sample(n=min(muestra_conteo, len(sub)), random_state=SEED)
    medidas = []
    for c in CRITERIOS:
        instr = instruccion_para(c)
        for _, f in filas_m.iterrows():
            try:
                r = cliente.models.count_tokens(
                    model=model_id, contents=f"{instr}\n\n{f['_payload']}"
                )
                medidas.append(r.total_tokens)
            except Exception:
                medidas.append(len(instr + f["_payload"]) // 4)   # respaldo heurístico
    tok_in_medio = sum(medidas) / max(len(medidas), 1)

    # Salida: justificación acotada + envoltura JSON del esquema
    tok_out_medio = JUSTIF_MAX_PALABRAS * 1.6 + 40
    if THINKING_BUDGET != 0:
        tok_out_medio += 500 if THINKING_BUDGET < 0 else THINKING_BUDGET

    llamadas = len(sub) * N_CRITERIOS
    t_in, t_out = llamadas * tok_in_medio, llamadas * tok_out_medio
    costo = t_in / 1e6 * p_in + t_out / 1e6 * p_out

    print(f"{'='*72}")
    print(f"ESTIMACIÓN · {model_id} · thinking={'apagado' if THINKING_BUDGET==0 else THINKING_BUDGET}")
    print(f"{'='*72}")
    print(f"  filas              {len(sub):,}")
    print(f"  criterios (N)      {N_CRITERIOS}")
    print(f"  llamadas           {llamadas:,}")
    print(f"  tokens entrada     {t_in/1e6:>8.2f} M  (medido: {tok_in_medio:,.0f}/llamada)")
    print(f"  tokens salida      {t_out/1e6:>8.2f} M  (estimado: {tok_out_medio:,.0f}/llamada)")
    print(f"  ─────────────────────────────────")
    print(f"  costo entrada      ${t_in/1e6*p_in:>8.2f}")
    print(f"  costo salida       ${t_out/1e6*p_out:>8.2f}   ← la salida domina")
    print(f"  COSTO TOTAL        ${costo:>8.2f}")
    print(f"  por criterio       ${costo/max(N_CRITERIOS,1):>8.2f}")
    print(f"  tiempo aprox.      {llamadas/CONCURRENCIA*2/60:>8.0f} min "
          f"(concurrencia {CONCURRENCIA})")
    print(f"{'='*72}")
    if THINKING_BUDGET != 0:
        print("  💡 THINKING_BUDGET=0 reduciría el costo ~2.7×")
    print(f"  💡 JUSTIF_MAX_PALABRAS={JUSTIF_MAX_PALABRAS} es la palanca principal")
    print(f"{'='*72}")
    return {"llamadas": llamadas, "costo": costo}

EST = estimar_costo(SUB)

CONFIRMO_EL_GASTO = False  #@param {type:"boolean"}
if not CONFIRMO_EL_GASTO:
    print("\n⛔ Revisa la estimación. Para autorizar la corrida, pon "
          "CONFIRMO_EL_GASTO = True y vuelve a ejecutar esta celda.")
else:
    print(f"\n✅ Gasto autorizado (~${EST.get('costo', 0):.2f}). Puedes correr la celda siguiente.")

In [ ]:
#@title 18 · Motor de corrida  ·  concurrencia + reintentos + checkpoint
import json, time, traceback
from pathlib import Path

def claves_completadas(path=CHECKPOINT_PATH) -> set:
    """Combinaciones (tweet_id, slug) ya presentes en el checkpoint."""
    p = Path(path)
    if not p.exists():
        return set()
    hechas = set()
    with p.open(encoding="utf-8") as fh:
        for linea in fh:
            linea = linea.strip()
            if not linea:
                continue
            try:
                r = json.loads(linea)
                hechas.add((str(r["tweet_id"]), r["slug"]))
            except Exception:
                continue   # línea truncada por una desconexión: se ignora
    return hechas

_ESCRITORES = {}

def _anexar(registros: list, path=CHECKPOINT_PATH) -> None:
    """Escritor por ruta: permite usar checkpoints separados (test-retest, calibración)
    sin que las líneas terminen en el archivo equivocado."""
    fh = _ESCRITORES.get(str(path))
    if fh is None or fh.closed:
        # Si una corrida anterior murió a media línea, el archivo no termina en \n.
        # Sin esto, el primer registro nuevo se concatenaría a la línea truncada
        # y se perdería también (JSON inválido).
        p = Path(path)
        if p.exists() and p.stat().st_size:
            with p.open("rb") as prev:
                prev.seek(-1, 2)
                cortada = prev.read(1) != b"\n"
            if cortada:
                with p.open("a", encoding="utf-8") as fix:
                    fix.write("\n")
                print(f"   ⚠️  '{path}' terminaba a media línea (desconexión previa): "
                      f"se cerró la línea; ese registro parcial se descarta al leer.")
        fh = open(path, "a", encoding="utf-8")
        _ESCRITORES[str(path)] = fh
    for r in registros:
        fh.write(json.dumps(r, ensure_ascii=False) + "\n")
    fh.flush()                 # ← inmediato: sobrevive a una desconexión

def _cerrar_escritor(path) -> None:
    fh = _ESCRITORES.pop(str(path), None)
    if fh and not fh.closed:
        fh.close()

def _motivo_fallo(exc: Exception) -> str:
    t = str(exc).lower()
    if any(k in t for k in ("safety", "blocked", "prohibited", "recitation")):
        return "BLOQUEADO"     # filtro de contenido: NO es nivel bajo ni no-aplicable
    if any(k in t for k in ("429", "rate limit", "resource_exhausted", "quota")):
        return "LIMITE_TASA"
    if any(k in t for k in ("500", "502", "503", "504", "unavailable", "internal")):
        return "ERROR_SERVIDOR"
    return "ERROR"

async def _evaluar_fila(runner, fila, pendientes: list, sem) -> list:
    tweet_id = str(fila[COL_ID])
    async with sem:
        ultimo = None
        for intento in range(1, MAX_INTENTOS + 1):
            try:
                claves = [f"cal_{s}" for s in pendientes]
                estado = await evaluar_payload(runner, fila["_payload"], claves)
                salida = []
                for slug in pendientes:
                    k = f"cal_{slug}"
                    if k in estado:
                        reg = validar_calificacion(slug, estado[k])
                    else:
                        reg = {"slug": slug, "aplicable": False, "nivel": None,
                               "puntaje": None, "justificacion": "",
                               "estado": "SIN_RESPUESTA", "detalle": "el agente no devolvió estado"}
                    reg["tweet_id"] = tweet_id
                    reg["contexto_incompleto"] = fila["contexto_incompleto"]
                    salida.append(reg)
                return salida
            except Exception as exc:
                ultimo = exc
                motivo = _motivo_fallo(exc)
                if motivo == "BLOQUEADO" or intento == MAX_INTENTOS:
                    break
                await asyncio.sleep(min(2 ** intento, 30))   # retroceso exponencial

        motivo = _motivo_fallo(ultimo)
        return [{"tweet_id": tweet_id, "slug": slug, "aplicable": False, "nivel": None,
                 "puntaje": None, "justificacion": "", "estado": motivo,
                 "detalle": str(ultimo)[:300],
                 "contexto_incompleto": fila["contexto_incompleto"]}
                for slug in pendientes]

async def correr(sub: pd.DataFrame, path=CHECKPOINT_PATH, equipo=None) -> dict:
    """`equipo` permite correr con un ParallelAgent distinto (p.ej. otro modelo en §6.4)
    sin mutar el equipo de producción."""
    equipo = equipo if equipo is not None else EQUIPO
    hechas = claves_completadas(path)
    todos_slugs = [c["slug"] for c in CRITERIOS]

    trabajo = []
    for _, fila in sub.iterrows():
        tid = str(fila[COL_ID])
        pend = [s for s in todos_slugs if (tid, s) not in hechas]
        if pend:
            trabajo.append((fila, pend))

    reusados = len(sub) * N_CRITERIOS - sum(len(p) for _, p in trabajo)
    print(f"Checkpoint '{path}': {reusados:,} resultados reutilizados")
    print(f"Pendientes: {sum(len(p) for _, p in trabajo):,} llamadas en {len(trabajo):,} filas")

    if not trabajo:
        print("✅ Nada pendiente: el checkpoint ya cubre todo el subconjunto.")
        return {"ok": 0, "fallidas": 0, "reusados": reusados}

    sem = asyncio.Semaphore(CONCURRENCIA)
    ok = fallidas = 0
    t0 = time.time()

    # Reciclado del runner por bloques: evita acumular sesiones en corridas largas
    for ini in range(0, len(trabajo), RECICLAR_CADA):
        bloque = trabajo[ini:ini + RECICLAR_CADA]
        runner = InMemoryRunner(agent=equipo, app_name=f"run_{ini}")

        tareas = [_evaluar_fila(runner, f, p, sem) for f, p in bloque]
        for fut in asyncio.as_completed(tareas):
            registros = await fut
            _anexar(registros, path)
            ok += sum(r["estado"] in ("OK", "NO_APLICABLE") for r in registros)
            fallidas += sum(r["estado"] not in ("OK", "NO_APLICABLE") for r in registros)

            hechas_n = ok + fallidas
            if hechas_n % 50 == 0 or hechas_n == sum(len(p) for _, p in trabajo):
                tasa = hechas_n / max(time.time() - t0, 1e-9)
                rest = (sum(len(p) for _, p in trabajo) - hechas_n) / max(tasa, 1e-9)
                print(f"  {hechas_n:,} listas · {fallidas:,} fallidas · "
                      f"{tasa:.1f}/s · faltan ~{rest/60:.0f} min", flush=True)

        del runner   # libera las sesiones del bloque

    print(f"\n✅ Corrida terminada en {(time.time()-t0)/60:.1f} min")
    print(f"   {ok:,} calificaciones · {fallidas:,} fallidas · {reusados:,} reutilizadas")
    if fallidas:
        print(f"   ⚠️  revisa los estados != OK en §6 para ver los motivos")
    return {"ok": ok, "fallidas": fallidas, "reusados": reusados}

print("Motor listo: correr(SUB)")

In [ ]:
#@title 19 · ▶️ Ejecutar la corrida
if not CONFIRMO_EL_GASTO:
    raise RuntimeError(
        "Gasto no autorizado. Vuelve a la celda de estimación, revisa el costo, "
        "pon CONFIRMO_EL_GASTO = True y ejecútala de nuevo antes de esta."
    )

RESUMEN = run_async(correr(SUB))

---
## § 6 · Verificaciones de calidad

Producir miles de calificaciones sin contraste no es un resultado, es un archivo. Cuatro mediciones, todas de costo marginal:

| Celda | Qué mide | Qué revela |
|---|---|---|
| Distribución | concentración de niveles por criterio | si un criterio no discrimina sobre tweets, o el descriptor se parseó mal del PDF |
| Test-retest | estabilidad al repetir | el piso de ruido del sistema; con `temperature=0` debería ser casi total |
| Acuerdo inter-modelo | `flash` contra `flash-lite` | criterios de juicio inestable. Aquí es donde el voting de `Ajuste Evaluador` aporta de verdad: como métrica sobre 100 filas, no como mecanismo de producción |
| Gold set | contra anotación humana | lo único que convierte "salieron números" en "salieron números confiables" |

In [ ]:
#@title 20 · Distribución de niveles por criterio  ·  el semáforo
def cargar_resultados(path=CHECKPOINT_PATH) -> pd.DataFrame:
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"No existe el checkpoint '{p}'.")
    filas = []
    with p.open(encoding="utf-8") as fh:
        for linea in fh:
            linea = linea.strip()
            if linea:
                try:
                    filas.append(json.loads(linea))
                except Exception:
                    continue
    df = pd.DataFrame(filas)
    # Si una fila se reevaluó, gana la última escrita
    return df.drop_duplicates(subset=["tweet_id", "slug"], keep="last")

def distribucion(res: pd.DataFrame = None) -> pd.DataFrame:
    res = res if res is not None else cargar_resultados()
    alertas = []
    print(f"{'='*74}")
    print(f"DISTRIBUCIÓN DE NIVELES · {res['tweet_id'].nunique():,} tweets × {N_CRITERIOS} criterios")
    print(f"{'='*74}")

    for c in CRITERIOS:
        sub = res[res["slug"] == c["slug"]]
        if sub.empty:
            print(f"\n[{c['id_criterio']}] {c['nombre'][:50]} — sin resultados")
            continue
        n = len(sub)
        n_na = (sub["estado"] == "NO_APLICABLE").sum()
        n_fail = (~sub["estado"].isin(["OK", "NO_APLICABLE"])).sum()
        ok = sub[sub["estado"] == "OK"]

        print(f"\n[{c['id_criterio']}] {c['nombre'][:50]}   (n={n:,})")
        if len(ok):
            conteo = ok["nivel"].value_counts()
            for etiqueta in c["etiquetas_validas"]:
                k = int(conteo.get(etiqueta, 0))
                prop = k / len(ok)
                barra = "▇" * int(prop * 34)
                print(f"    {etiqueta[:26]:<26} {barra:<34} {k:>6,} {prop:>6.1%}")
            top = conteo.iloc[0] / len(ok)
            if top > UMBRAL_CONCENTRACION:
                alertas.append(
                    f"[{c['id_criterio']}] SIN PODER DISCRIMINANTE: {top:.0%} en «{conteo.index[0]}». "
                    f"O el criterio no aplica a tweets, o el descriptor se parseó mal, o el prompt sesga."
                )
        print(f"    {'no aplicable':<26} {'▁'*int(n_na/n*34):<34} {n_na:>6,} {n_na/n:>6.1%}")
        if n_fail:
            motivos = sub[~sub["estado"].isin(["OK","NO_APLICABLE"])]["estado"].value_counts().to_dict()
            print(f"    {'fallidas':<26} {'':<34} {n_fail:>6,} {n_fail/n:>6.1%}  {motivos}")
        if n_na / n > UMBRAL_NO_APLICABLE:
            alertas.append(
                f"[{c['id_criterio']}] MAYORITARIAMENTE NO APLICABLE ({n_na/n:.0%}): "
                f"la rúbrica puede no corresponder a este tipo de texto."
            )

    print(f"\n{'='*74}")
    if alertas:
        print("⚠️  ALERTAS")
        for a in alertas:
            print(f"   · {a}")
    else:
        print("✅ Ningún criterio supera los umbrales de concentración ni de no aplicabilidad.")
    print(f"{'='*74}")
    return res

RES = distribucion()

In [ ]:
#@title 21 · Test-retest  ·  estabilidad al repetir  (checkpoint separado)
CKPT_RETEST = "checkpoint_retest.jsonl"

def test_retest(n: int = 60) -> pd.DataFrame:
    """Reevalúa una muestra en un checkpoint APARTE. El original queda intacto."""
    muestra = SUB.sample(n=min(n, len(SUB)), random_state=SEED + 1)
    print(f"Reevaluando {len(muestra)} filas × {N_CRITERIOS} criterios en '{CKPT_RETEST}'…")

    _cerrar_escritor(CKPT_RETEST)
    Path(CKPT_RETEST).unlink(missing_ok=True)
    try:
        run_async(correr(muestra, path=CKPT_RETEST))
    finally:
        _cerrar_escritor(CKPT_RETEST)

    orig = cargar_resultados(CHECKPOINT_PATH)
    re_  = cargar_resultados(CKPT_RETEST)
    j = orig.merge(re_, on=["tweet_id", "slug"], suffixes=("_1", "_2"))

    print(f"\n{'='*66}\nESTABILIDAD (temperature={TEMPERATURE}, thinking={THINKING_BUDGET})\n{'='*66}")
    filas = []
    for c in CRITERIOS:
        s = j[j["slug"] == c["slug"]]
        if s.empty:
            continue
        acuerdo = (s["nivel_1"].fillna("∅") == s["nivel_2"].fillna("∅")).mean()
        marca = "✅" if acuerdo >= UMBRAL_ESTABILIDAD else "⚠️"
        print(f"  {marca} [{c['id_criterio']}] {c['nombre'][:42]:<42} acuerdo {acuerdo:>6.1%} (n={len(s)})")
        filas.append({"criterio": c["id_criterio"], "acuerdo": acuerdo, "n": len(s)})

    if filas:
        glob = sum(f["acuerdo"] * f["n"] for f in filas) / sum(f["n"] for f in filas)
        print(f"\n  Acuerdo global: {glob:.1%}")
        if glob < UMBRAL_ESTABILIDAD:
            print(f"  ⚠️  Por debajo de {UMBRAL_ESTABILIDAD:.0%}: hay más ruido del esperado "
                  f"para una configuración determinista. Revisa los criterios marcados.")
    print(f"{'='*66}")
    # El checkpoint original no se tocó
    assert len(cargar_resultados(CHECKPOINT_PATH)) == len(orig), "El checkpoint original cambió"
    return pd.DataFrame(filas)

# RETEST = test_retest(60)   ← descomenta para ejecutar (cuesta ~N × $0.05)
print("Listo: test_retest(60)")

In [ ]:
#@title 22 · Acuerdo inter-modelo  ·  calibración, no producción
CKPT_CALIB = "checkpoint_calibracion.jsonl"

def acuerdo_inter_modelo(n: int = 60, model_id: str = None) -> pd.DataFrame:
    """Evalúa una muestra con un segundo modelo y compara. NO es el mecanismo de producción:
    en producción se usa un solo modelo por criterio."""
    model_id = model_id or MODEL_ID_CALIBRACION
    muestra = SUB.sample(n=min(n, len(SUB)), random_state=SEED + 2)
    print(f"Calibrando con '{model_id}' sobre {len(muestra)} filas…")

    # Equipo temporal con el otro modelo. No se toca EQUIPO: se pasa por argumento.
    equipo_calib, _ = construir_equipo(model_id)
    _cerrar_escritor(CKPT_CALIB)
    Path(CKPT_CALIB).unlink(missing_ok=True)
    try:
        run_async(correr(muestra, path=CKPT_CALIB, equipo=equipo_calib))
    finally:
        _cerrar_escritor(CKPT_CALIB)

    a = cargar_resultados(CHECKPOINT_PATH)
    b = cargar_resultados(CKPT_CALIB)
    j = a.merge(b, on=["tweet_id", "slug"], suffixes=("_a", "_b"))

    orden = {c["slug"]: {e: i for i, e in enumerate(c["etiquetas_validas"])} for c in CRITERIOS}
    print(f"\n{'='*72}\nACUERDO {MODEL_ID}  vs  {model_id}\n{'='*72}")
    filas = []
    for c in CRITERIOS:
        s = j[j["slug"] == c["slug"]]
        if s.empty:
            continue
        exacto = (s["nivel_a"].fillna("∅") == s["nivel_b"].fillna("∅")).mean()
        o = orden[c["slug"]]
        dist1 = s.apply(
            lambda r: (r["nivel_a"] in o and r["nivel_b"] in o
                       and abs(o[r["nivel_a"]] - o[r["nivel_b"]]) <= 1), axis=1
        ).mean()
        marca = "⚠️" if exacto < 0.6 else "✅"
        print(f"  {marca} [{c['id_criterio']}] {c['nombre'][:38]:<38} "
              f"exacto {exacto:>6.1%} · ±1 nivel {dist1:>6.1%} (n={len(s)})")
        if exacto < 0.6:
            print(f"       ⚠️  juicio inestable: candidato a reformular el descriptor del criterio")
        filas.append({"criterio": c["id_criterio"], "exacto": exacto, "pm1": dist1, "n": len(s)})
    print(f"{'='*72}")
    return pd.DataFrame(filas)

# CALIB = acuerdo_inter_modelo(60)   ← descomenta para ejecutar
print(f"Listo: acuerdo_inter_modelo(60)  ·  segundo modelo: {MODEL_ID_CALIBRACION}")

In [ ]:
#@title 23 · Gold set  ·  contraste contra anotación humana
GOLD_PATH = "gold_set.csv"   # columnas: tweet_id, slug, nivel_humano

def plantilla_gold(n: int = 50, path: str = GOLD_PATH) -> None:
    """Genera una plantilla para anotar a mano: una fila por tweet × criterio."""
    muestra = SUB.sample(n=min(n, len(SUB)), random_state=SEED + 3)
    filas = [
        {"tweet_id": str(f[COL_ID]), "slug": c["slug"], "id_criterio": c["id_criterio"],
         "criterio": c["nombre"], "niveles_validos": " | ".join(c["etiquetas_validas"]),
         "texto": str(f[COL_TEXTO])[:400], "nivel_humano": ""}
        for _, f in muestra.iterrows() for c in CRITERIOS
    ]
    pd.DataFrame(filas).to_csv(path, index=False, encoding="utf-8-sig")
    print(f"✅ Plantilla escrita en '{path}' ({len(filas):,} filas para anotar).")
    print("   Llena la columna 'nivel_humano' con una de las etiquetas válidas, "
          "déjala vacía si el criterio no aplica, y vuelve a subir el archivo.")

def comparar_gold(path: str = GOLD_PATH) -> pd.DataFrame:
    p = Path(path)
    if not p.exists():
        print(f"⚠️  No hay gold set en '{path}'.")
        print("   Las calificaciones NO TIENEN CONTRASTE CON JUICIO HUMANO. La distribución y "
              "la estabilidad dicen si el sistema es consistente, no si tiene razón.")
        print(f"   Genera una plantilla con: plantilla_gold(50)")
        return pd.DataFrame()

    gold = pd.read_csv(p, dtype=str).dropna(subset=["nivel_humano"])
    gold = gold[gold["nivel_humano"].str.strip() != ""]
    if gold.empty:
        print(f"⚠️  '{path}' existe pero no tiene ninguna anotación en 'nivel_humano'.")
        return pd.DataFrame()

    res = cargar_resultados()
    j = gold.merge(res, on=["tweet_id", "slug"], how="inner")
    print(f"{'='*72}\nACUERDO CON ANOTACIÓN HUMANA  (n={len(j):,})\n{'='*72}")
    filas = []
    for c in CRITERIOS:
        s = j[j["slug"] == c["slug"]]
        if s.empty:
            continue
        acuerdo = (s["nivel_humano"].str.strip() == s["nivel"].fillna("")).mean()
        marca = "✅" if acuerdo >= 0.7 else "⚠️"
        print(f"  {marca} [{c['id_criterio']}] {c['nombre'][:44]:<44} acuerdo {acuerdo:>6.1%} (n={len(s)})")
        filas.append({"criterio": c["id_criterio"], "acuerdo": acuerdo, "n": len(s)})

    desac = j[j["nivel_humano"].str.strip() != j["nivel"].fillna("")]
    if len(desac):
        print(f"\n── DESACUERDOS ({len(desac)}) ──")
        for _, r in desac.head(15).iterrows():
            print(f"\n  [{r['id_criterio']}] tweet {r['tweet_id']}")
            print(f"    humano: {r['nivel_humano']}   |   sistema: {r['nivel']}")
            print(f"    justificación del sistema: {str(r['justificacion'])[:220]}")
    print(f"\n{'='*72}")
    return pd.DataFrame(filas)

GOLD = comparar_gold()

---
## § 7 · Exportación

Dos archivos:

- **Ancho** — el CSV original **íntegro** (las 52 columnas, todas las filas) más `{criterio}_nivel`, `{criterio}_puntaje`, `{criterio}_justificacion` y `{criterio}_aplicable` por criterio. Las columnas originales no se tocan, incluidas `tipo` y `justificacion`, que quedan vacías como estaban: dos columnas no alcanzan para N criterios y pueden tener otro dueño.
- **Tidy** — una fila por tweet × criterio, para análisis.

Ambos en **UTF-8 con BOM**. Sin el BOM, Excel destroza los acentos de las justificaciones; el problema ya está documentado dentro de `Ajuste Evaluador`, cuyo `criteria_prod.json` tiene mojibake.

In [ ]:
#@title 24 · Merge al CSV original  +  verificación de integridad
def nombres_columnas(criterio: dict) -> dict:
    s = criterio["slug"]
    cols = {"nivel": f"{s}_nivel", "justificacion": f"{s}_justificacion",
            "aplicable": f"{s}_aplicable"}
    if criterio["escala_numerica"]:
        cols["puntaje"] = f"{s}_puntaje"      # sólo si la rúbrica define puntos
    return cols

def construir_salidas(df_original: pd.DataFrame, res: pd.DataFrame = None) -> tuple:
    res = res if res is not None else cargar_resultados()

    # ── Integridad: resultados cuyo tweet no existe en el corpus ──────────
    ids_corpus = set(df_original[COL_ID].astype(str))
    huerfanos = sorted(set(res["tweet_id"]) - ids_corpus)
    if huerfanos:
        print(f"⚠️  {len(huerfanos)} resultados con tweet_id inexistente en el corpus "
              f"(no se incorporan): {huerfanos[:5]}")
        res = res[res["tweet_id"].isin(ids_corpus)]

    # ── Ancho ─────────────────────────────────────────────────────────────
    ancho = df_original.copy()
    ancho["_k"] = ancho[COL_ID].astype(str)
    n_cols = 0
    for c in CRITERIOS:
        cols = nombres_columnas(c)
        s = res[res["slug"] == c["slug"]].drop_duplicates(subset=["tweet_id"], keep="last")
        m = s.set_index("tweet_id")
        for campo, nombre in cols.items():
            if campo != "aplicable" and campo not in m.columns:
                ancho[nombre] = pd.NA      # ningún resultado trajo este campo
                n_cols += 1
                continue
            if campo == "aplicable":
                # texto explícito: distingue no evaluada (vacío) de fallida
                serie = m["estado"].map(
                    lambda e: "" if pd.isna(e) else
                              ("SI" if e == "OK" else ("NO" if e == "NO_APLICABLE" else e))
                )
            else:
                serie = m[campo]
            ancho[nombre] = ancho["_k"].map(serie)
            n_cols += 1
    ancho = ancho.drop(columns=["_k"])

    # ── Verificación de conteo de filas ───────────────────────────────────
    if len(ancho) != len(df_original):
        raise RuntimeError(
            f"El merge alteró el número de filas: {len(df_original):,} → {len(ancho):,}. "
            f"Probable duplicación de '{COL_ID}'."
        )
    dups = df_original[COL_ID].astype(str).duplicated().sum()
    if dups:
        print(f"⚠️  {dups:,} '{COL_ID}' duplicados en el original: esas filas comparten calificación")

    # ── Tidy ──────────────────────────────────────────────────────────────
    meta = {c["slug"]: (c["id_criterio"], c["nombre"]) for c in CRITERIOS}
    tidy = res.copy()
    tidy["id_criterio"] = tidy["slug"].map(lambda s: meta.get(s, ("", ""))[0])
    tidy["criterio"] = tidy["slug"].map(lambda s: meta.get(s, ("", ""))[1])
    tidy = tidy.reindex(columns=["tweet_id", "id_criterio", "slug", "criterio", "aplicable",
                                 "nivel", "puntaje", "justificacion", "estado", "detalle",
                                 "contexto_incompleto"])

    evaluadas = ancho[nombres_columnas(CRITERIOS[0])["nivel"]].notna().sum() if CRITERIOS else 0
    print(f"\n✅ Ancho: {len(ancho):,} filas × {len(ancho.columns)} columnas "
          f"({n_cols} añadidas, {len(df_original.columns)} originales intactas)")
    print(f"   filas con calificación: {evaluadas:,} · "
          f"sin evaluar (columnas vacías): {len(ancho) - evaluadas:,}")
    print(f"✅ Tidy : {len(tidy):,} filas (tweet × criterio)")
    return ancho, tidy

ANCHO, TIDY = construir_salidas(DF)
ANCHO.head(3)

In [ ]:
#@title 25 · Escribir con utf-8-sig  y  descargar
def exportar(ancho: pd.DataFrame, tidy: pd.DataFrame, descargar: bool = True) -> list:
    destinos = [Path(".")]
    if SALIDA_DIR_DRIVE:
        montar_drive_si_hace_falta(SALIDA_DIR_DRIVE)
        d = Path(SALIDA_DIR_DRIVE)
        d.mkdir(parents=True, exist_ok=True)
        destinos.append(d)

    escritos = []
    for d in destinos:
        for df, nombre in ((ancho, SALIDA_ANCHO), (tidy, SALIDA_TIDY)):
            ruta = d / nombre
            # utf-8-sig: el BOM es lo que hace que Excel respete los acentos
            df.to_csv(ruta, index=False, encoding="utf-8-sig")
            escritos.append(ruta)
            print(f"✅ {ruta}  ({ruta.stat().st_size/1e6:.1f} MB)")

    # Comprobación de ida y vuelta de la codificación
    prueba = pd.read_csv(escritos[0], dtype=str, nrows=200, encoding="utf-8-sig")
    col_j = next((c for c in prueba.columns if c.endswith("_justificacion")), None)
    if col_j is not None:
        textos = prueba[col_j].dropna().astype(str)
        acentos = textos[textos.str.contains(r"[áéíóúñÁÉÍÓÚÑ¿¡]", na=False)]
        print(f"\n   codificación: {len(acentos)} justificaciones con acentos releídas correctamente")
        if len(acentos):
            print(f"   muestra: {acentos.iloc[0][:120]}")
    if COL_TEXTO in prueba.columns:
        cjk = prueba[COL_TEXTO].dropna().astype(str)
        cjk = cjk[cjk.str.contains(r"[\u3040-\u30ff\u4e00-\u9fff\uac00-\ud7af]", na=False)]
        if len(cjk):
            print(f"   caracteres no latinos preservados: {cjk.iloc[0][:60]}")

    print(f"\n{'='*66}")
    print(f"RESUMEN DE EXPORTACIÓN")
    print(f"  filas          {len(ancho):,}")
    print(f"  columnas       {len(ancho.columns)} ({len(ancho.columns)-len(DF.columns)} añadidas)")
    print(f"  criterios      {N_CRITERIOS}")
    print(f"  filas tidy     {len(tidy):,}")
    print(f"{'='*66}")

    if descargar:
        try:
            from google.colab import files
            for r in escritos[:2]:
                files.download(str(r))
        except Exception as exc:
            print(f"(descarga automática no disponible: {exc})")
    return escritos

RUTAS = exportar(ANCHO, TIDY)

---
## § 8 · Visualización

Las verificaciones de la § 6 dicen si las calificaciones son **consistentes**; no dicen qué **muestran**. La pregunta para la que se armó el corpus —si la percepción de México como destino se movió alrededor de los partidos del mundial en el país— es temporal y comparativa, y ninguna celda anterior la aborda.

Tres familias de gráficas, todas sobre el mismo agregado:

```
volumen diario del corpus        ← el denominador: explica los saltos de las demás
serie diaria por criterio        ← evolución de los niveles, con los períodos marcados
distribución global y por período ← la comparación antes / durante / después
```

**Períodos.** Definidos en `CONFIG` como intervalos `[desde, hasta)`. El corte del 6 de julio no es arbitrario: el último partido jugado en territorio mexicano fueron los octavos del 5 de julio en el Estadio Azteca. El torneo siguió hasta el 19 de julio fuera de México, de ahí el nombre del tercer período.

**El día se corta en hora de México**, no en UTC. Los cortes son eventos locales y el público que se mide vive ese calendario. Medido sobre este corpus: 602 tuits, el 23.5%, cambian de día al convertir.

**Se agrega en pandas y se filtra en el navegador.** La celda calcula los conteos por `(fecha, criterio, nivel, idioma)` una sola vez y los embebe en la página; el filtro de idioma y el conmutador de modo no vuelven a pasar por Python. El resultado es un HTML autocontenido que sigue funcionando fuera del notebook.

⚠️ **Requiere una corrida sobre el corpus completo.** Con `N_MUESTRA = 200` son ~3 tuits por día repartidos entre los criterios: la serie diaria dibujaría la semilla del muestreo, no el fenómeno. La celda lo advierte cuando lo detecta.


In [ ]:
#@title 26 · Preparación  ·  unión con el corpus, fecha local y período
from datetime import date

ETIQUETA_SIN_IDIOMA = "(sin idioma)"

# Códigos que la API devuelve cuando no hay texto del cual inferir idioma.
# No son idiomas, y conviene que quien use el filtro lo sepa antes de elegirlos:
# 'und' indeterminado, 'zxx' sin contenido lingüístico, 'qme'/'qam'/'qct'/'qht'/
# 'qst' marcadores internos de contenido multimedia o mixto, 'art' artificial.
CODIGOS_SIN_IDIOMA = {"und", "zxx", "qme", "qam", "qct", "qht", "qst", "art"}


def _a_utc(serie: pd.Series) -> pd.Series:
    """El corpus mezcla '2026-06-03T23:59:50' con '2026-07-04 7:02:11'.
    Las marcas sin huso se interpretan como UTC, que es lo que entrega la API."""
    for extra in ({"format": "mixed"}, {"format": "ISO8601"}, {}):
        try:
            return pd.to_datetime(serie, errors="coerce", utc=True, **extra)
        except (ValueError, TypeError):
            continue
    return pd.to_datetime(serie, errors="coerce", utc=True)


def _a_hora_local(dt_utc: pd.Series) -> tuple:
    """Devuelve (serie local, zona efectiva). Si la zona no existe en el entorno
    se informa y se sigue en UTC: es preferible una gráfica con el día corrido
    seis horas, dicha en voz alta, que ninguna gráfica."""
    try:
        return dt_utc.dt.tz_convert(ZONA_HORARIA), ZONA_HORARIA
    except Exception as exc:
        print(f"   ⚠️  no se pudo convertir a '{ZONA_HORARIA}' ({exc}). "
              f"Se agrupa en UTC: los cortes de período quedan corridos hasta 6 h.")
        return dt_utc, "UTC"


def asignar_periodo(fecha_local: pd.Series) -> pd.Series:
    """Intervalos [desde, hasta). El primero que acepta la fecha se la queda,
    y CONFIG ya garantizó que no se solapan, así que el orden no decide nada."""
    periodo = pd.Series(pd.NA, index=fecha_local.index, dtype="object")
    for p in PERIODOS:
        dentro = fecha_local >= pd.Timestamp(p["desde"])
        if p["hasta"] is not None:
            dentro &= fecha_local < pd.Timestamp(p["hasta"])
        periodo = periodo.mask(dentro & periodo.isna(), p["nombre"])
    return periodo


def _fuente_calificaciones() -> pd.DataFrame:
    """TIDY si la sesión lo tiene; si no, el CSV tidy exportado; si no, el
    checkpoint. Permite graficar sin reejecutar el notebook entero."""
    if isinstance(globals().get("TIDY"), pd.DataFrame):
        return globals()["TIDY"].copy()
    if Path(SALIDA_TIDY).exists():
        print(f"   ℹ️  'TIDY' no está en memoria; se lee '{SALIDA_TIDY}'.")
        return pd.read_csv(SALIDA_TIDY, dtype=str, keep_default_na=False, na_values=[""])
    print(f"   ℹ️  'TIDY' no está en memoria; se reconstruye desde '{CHECKPOINT_PATH}'.")
    res = cargar_resultados()
    meta = {c["slug"]: (c["id_criterio"], c["nombre"]) for c in CRITERIOS}
    res["id_criterio"] = res["slug"].map(lambda s: meta.get(s, ("", ""))[0])
    res["criterio"] = res["slug"].map(lambda s: meta.get(s, ("", ""))[1])
    return res


def _fuente_corpus() -> pd.DataFrame:
    if isinstance(globals().get("DF"), pd.DataFrame):
        return globals()["DF"]
    print(f"   ℹ️  'DF' no está en memoria; se lee '{CSV_PATH}'.")
    return cargar_csv()


def preparar_visualizacion(tidy: pd.DataFrame = None, df: pd.DataFrame = None) -> pd.DataFrame:
    """Une calificaciones con corpus y añade fecha local, idioma y período.

    TIDY sale con tweet_id, slug, nivel, estado y poco más: no trae ni fecha ni
    idioma. Toda gráfica de esta sección empieza por esta unión."""
    tidy = tidy if tidy is not None else _fuente_calificaciones()
    df = df if df is not None else _fuente_corpus()
    print("=" * 74)
    print(f"PREPARACIÓN · {len(tidy):,} calificaciones × {len(df):,} filas de corpus")
    print("=" * 74)

    faltan = [c for c in ("tweet_id", "slug", "nivel", "estado") if c not in tidy.columns]
    if faltan:
        raise RuntimeError(f"Las calificaciones no traen {faltan}. ¿Corriste la § 5?")

    cols = [COL_ID, "created_at", COL_LANG]
    ausentes = [c for c in cols if c not in df.columns]
    if ausentes:
        raise RuntimeError(
            f"El corpus no trae {ausentes}. Sin 'created_at' no hay eje temporal "
            f"y sin '{COL_LANG}' no hay filtro de idioma."
        )

    corpus = df[cols].copy()
    corpus[COL_ID] = corpus[COL_ID].astype(str).str.strip()
    corpus = corpus.drop_duplicates(subset=[COL_ID], keep="first")

    viz = tidy.copy()
    viz["tweet_id"] = viz["tweet_id"].astype(str).str.strip()

    # ── Huérfanos: el mismo criterio de integridad que el merge de § 7 ────
    ids_corpus = set(corpus[COL_ID])
    fuera = ~viz["tweet_id"].isin(ids_corpus)
    if fuera.any():
        n_ids = viz.loc[fuera, "tweet_id"].nunique()
        print(f"   ⚠️  {int(fuera.sum()):,} calificaciones de {n_ids:,} identificadores "
              f"inexistentes en el corpus — excluidas")
        print(f"       ejemplos: {sorted(viz.loc[fuera, 'tweet_id'].unique())[:3]}")
        if n_ids == viz["tweet_id"].nunique():
            print("       ⛔ NINGÚN identificador coincide. Es la firma de un corpus sin "
                  "reparar: los de 19 dígitos pasaron por float64 y ya no son ellos. "
                  "Corre la celda 8.5 antes de graficar.")
        viz = viz[~fuera]

    viz = viz.merge(corpus, left_on="tweet_id", right_on=COL_ID,
                    how="left", suffixes=("", "_corpus"))

    # ── Fecha: UTC → hora local → día ────────────────────────────────────
    dt_utc = _a_utc(viz["created_at"])
    ilegible = dt_utc.isna()
    if ilegible.any():
        print(f"   ⚠️  {int(ilegible.sum()):,} calificaciones con 'created_at' "
              f"ilegible — excluidas")
        viz, dt_utc = viz[~ilegible], dt_utc[~ilegible]

    dt_local, zona = _a_hora_local(dt_utc)
    viz["fecha"] = dt_local.dt.normalize().dt.tz_localize(None).values
    viz["fecha_str"] = viz["fecha"].dt.strftime("%Y-%m-%d")

    # ── Idioma ───────────────────────────────────────────────────────────
    lang = viz[COL_LANG].fillna("").astype(str).str.strip()
    viz["idioma"] = lang.where(lang != "", ETIQUETA_SIN_IDIOMA)

    # ── Autosuficiencia: replies, retweets y quotes no lo son ────────────
    marca = viz["contexto_incompleto"] if "contexto_incompleto" in viz.columns else None
    if marca is None:
        print("   ℹ️  sin columna 'contexto_incompleto': todo se cuenta como autosuficiente")
        viz["autosuficiente"] = True
    else:
        viz["autosuficiente"] = marca.fillna("").astype(str).str.strip() == ""

    # ── Período ──────────────────────────────────────────────────────────
    viz["periodo"] = asignar_periodo(viz["fecha"])
    sin_periodo = viz["periodo"].isna()
    if sin_periodo.any():
        rango = viz.loc[sin_periodo, "fecha_str"]
        print(f"   ⚠️  {int(sin_periodo.sum()):,} calificaciones fuera de todo período "
              f"({rango.min()} … {rango.max()}) — quedan en la serie diaria, "
              f"no en los paneles por período")

    # ── Reporte ──────────────────────────────────────────────────────────
    n_tweets = viz["tweet_id"].nunique()
    dias = viz["fecha_str"].nunique()
    por_dia = n_tweets / max(dias, 1)
    n_incompletos = int((~viz["autosuficiente"]).sum())
    print(f"\n   {len(viz):,} calificaciones · {n_tweets:,} tuits · {dias} días "
          f"({viz['fecha_str'].min()} … {viz['fecha_str'].max()})")
    print(f"   día cortado en {zona} · contexto incompleto: {n_incompletos:,} "
          f"calificaciones ({n_incompletos / max(len(viz), 1):.1%})")

    print(f"\n   {'período':<32} {'días':>5} {'tuits':>8} {'por día':>8}")
    for p in PERIODOS:
        s = viz[viz["periodo"] == p["nombre"]]
        d, n = s["fecha_str"].nunique(), s["tweet_id"].nunique()
        print(f"   {p['nombre']:<32} {d:>5} {n:>8,} {n / max(d, 1):>8.1f}")

    if por_dia < UMBRAL_COBERTURA_DIARIA:
        print(f"\n   ⚠️  COBERTURA INSUFICIENTE: {por_dia:.1f} tuits calificados por día, "
              f"bajo el umbral de {UMBRAL_COBERTURA_DIARIA}.")
        print("       Repartidos entre los niveles de un criterio, cada punto de la serie "
              "diaria vale menos de un tuit: lo que se vería es la semilla del muestreo, "
              "no el fenómeno. Pon N_MUESTRA = None y corre el corpus completo.")
    print("=" * 74)
    return viz


VIZ = preparar_visualizacion()
VIZ.head(3)


In [ ]:
#@title 27 · Agregación  ·  conteos por día, criterio, nivel e idioma
import json, re
import numpy as np

# ── Qué nivel significa "ausencia" ───────────────────────────────────────
# Esta rúbrica resuelve el problema del nivel 0 con una escala híbrida: el 0 es
# AUSENCIA (el tuit no toca la dimensión) y 1-5 es valencia, de muy negativa a
# muy positiva. El 0 no es ordinalmente contiguo al 1 —pasar de "ausencia" a
# "muy negativo" no es un incremento en la misma latente— así que apilarlo con
# los demás lo pondría en el extremo negativo, que es lo contrario de lo que
# significa. Se separa: valencia por un lado, ausencia por otro.
_NEGACION = re.compile(
    r"\bno\s+(evalúa|evalua|expresa|menciona|contiene|aborda|refiere|toca|hay)\b",
    re.IGNORECASE)


def _detectar_niveles_ausencia(criterio: dict) -> set:
    """Un nivel es de ausencia si su descriptor niega que el tuit toque la
    dimensión. Deliberadamente conservador: si la rúbrica no lo dice así,
    devuelve vacío y todos los niveles se grafican."""
    encontrados = set()
    for n in criterio["niveles"]:
        if _NEGACION.search(str(n.get("descriptor", ""))[:160]):
            encontrados.add(str(n["etiqueta"]))
    return encontrados


def resolver_niveles_ausencia(criterios: list = None) -> dict:
    """NIVELES_AUSENCIA acepta 'auto', None, una lista de etiquetas para todos,
    o un dict {slug: [etiquetas]}."""
    criterios = criterios if criterios is not None else CRITERIOS
    cfg = globals().get("NIVELES_AUSENCIA", "auto")
    mapa = {}
    for c in criterios:
        if cfg == "auto":
            mapa[c["slug"]] = _detectar_niveles_ausencia(c)
        elif not cfg:
            mapa[c["slug"]] = set()
        elif isinstance(cfg, dict):
            mapa[c["slug"]] = {str(x) for x in cfg.get(c["slug"], [])}
        else:
            mapa[c["slug"]] = {str(x) for x in cfg}
    return mapa


# ── Agregación ───────────────────────────────────────────────────────────
# Se agrega ANTES de dibujar y se embebe sólo el agregado: el filtro de idioma
# y el conmutador absoluto/proporción son reducciones sobre un arreglo chico en
# el cliente, no una reejecución de Python. Las combinaciones vacías se omiten:
# la matriz completa (días × criterios × niveles × idiomas) es casi toda hueca.
def agregar(viz: pd.DataFrame = None) -> dict:
    viz = viz if viz is not None else VIZ
    ausencia = resolver_niveles_ausencia()

    v = viz.copy()
    v["nivel"] = v["nivel"].astype(str).str.strip()
    v["estado"] = v["estado"].astype(str).str.strip()
    es_ausencia = v.apply(
        lambda r: r["estado"] == "NO_APLICABLE"
        or (r["estado"] == "OK" and r["nivel"] in ausencia.get(r["slug"], set())),
        axis=1)
    v["clase"] = np.where(
        es_ausencia, "ausencia",
        np.where(v["estado"] == "OK", "valencia", "fallida"))
    v.loc[v["clase"] != "valencia", "nivel"] = ""

    # Conteos por día · criterio · nivel · idioma · autosuficiencia
    agg = (v.groupby(["fecha_str", "slug", "clase", "nivel", "idioma",
                      "autosuficiente"], dropna=False)
             .size().reset_index(name="n"))
    agg = agg[agg["n"] > 0]

    # Volumen del corpus: tuits distintos, no calificaciones
    vol = (v.drop_duplicates(subset=["tweet_id", "fecha_str", "idioma", "autosuficiente"])
             .groupby(["fecha_str", "idioma", "autosuficiente"])
             .size().reset_index(name="n"))

    # Totales por idioma, para mostrarlos junto a cada opción del filtro
    por_idioma = (v.drop_duplicates(subset=["tweet_id"])
                    .groupby("idioma").size().sort_values(ascending=False))

    # Totales por período
    periodos = []
    for p in PERIODOS:
        s = v[v["periodo"] == p["nombre"]]
        d = int(s["fecha_str"].nunique())
        periodos.append({"nombre": p["nombre"], "dias": d,
                         "tuits": int(s["tweet_id"].nunique()),
                         "por_dia": round(s["tweet_id"].nunique() / max(d, 1), 1)})

    orden = {c["slug"]: [str(e) for e in c["etiquetas_validas"]] for c in CRITERIOS}
    criterios = [{"slug": c["slug"], "id": c["id_criterio"], "nombre": c["nombre"],
                  "niveles": [e for e in orden[c["slug"]]
                              if e not in ausencia.get(c["slug"], set())],
                  "ausencia": sorted(ausencia.get(c["slug"], set()))}
                 for c in CRITERIOS]

    fechas = sorted(v["fecha_str"].dropna().unique())
    datos = {
        "criterios": criterios,
        "fechas": fechas,
        "periodos": periodos,
        "idiomas": [{"codigo": k, "n": int(n), "sin_idioma": k in CODIGOS_SIN_IDIOMA
                     or k == ETIQUETA_SIN_IDIOMA} for k, n in por_idioma.items()],
        "agg": agg.to_dict("records"),
        "volumen": vol.to_dict("records"),
        "umbral_muestra": UMBRAL_MUESTRA_PEQUENA,
        "ventana": VENTANA_SUAVIZADO,
        "cortes": [{"nombre": p["nombre"], "desde": str(p["desde"]),
                    "hasta": str(p["hasta"]) if p["hasta"] else None} for p in PERIODOS],
    }

    # ── Reporte ──────────────────────────────────────────────────────────
    bytes_json = len(json.dumps(datos, ensure_ascii=False).encode("utf-8"))
    print("=" * 74)
    print("AGREGACIÓN")
    print("=" * 74)
    for c in criterios:
        s = v[v["slug"] == c["slug"]]
        val = int((s["clase"] == "valencia").sum())
        aus = int((s["clase"] == "ausencia").sum())
        fal = int((s["clase"] == "fallida").sum())
        tot = max(len(s), 1)
        print(f"\n  [{c['id']}] {c['nombre'][:48]}")
        print(f"      valencia {val:>6,} ({val/tot:>5.1%})  ·  "
              f"ausencia {aus:>6,} ({aus/tot:>5.1%})  ·  fallidas {fal:>4,}")
        if c["ausencia"]:
            print(f"      nivel(es) de ausencia detectados: {c['ausencia']} "
                  f"— fuera de las series y del denominador")
        else:
            print(f"      sin nivel de ausencia en la rúbrica: se grafican todos")

    print(f"\n  agregado: {len(agg):,} filas · volumen: {len(vol):,} filas")
    print(f"  payload serializado: {bytes_json/1024:.0f} KB")
    print(f"  {len(fechas)} días · {len(datos['idiomas'])} idiomas · "
          f"{len(criterios)} criterios")
    print(f"\n  {'período':<32} {'días':>5} {'tuits':>8} {'por día':>8}")
    for p in periodos:
        print(f"  {p['nombre']:<32} {p['dias']:>5} {p['tuits']:>8,} {p['por_dia']:>8.1f}")
    suma = sum(p["tuits"] for p in periodos)
    print(f"  {'suma':<32} {'':>5} {suma:>8,}")
    print("=" * 74)
    return datos


DATOS = agregar()


In [ ]:
#@title 28 · Gráficas  ·  ECharts, filtrables, dentro del Colab y como .html
import json
from pathlib import Path

# ── Paleta ───────────────────────────────────────────────────────────────
# Valencia: divergente rojo↔azul con gris neutro al centro. NO es una escala de
# logro: el 1 es "muy negativo" y el 5 "muy positivo", así que un solo tono
# claro→oscuro mentiría sobre la polaridad. Cada brazo validado como rampa
# ordinal (lightness monótona, separación ≥0.06, extremo claro sobre el fondo).
# Idiomas: categórica nominal, sin azul ni rojo para que no se confundan con
# la valencia. Ambas pasan el validador en el pairlist adyacente.
PALETA_VALENCIA = ["#c62f2e", "#e8716f", "#c3c2b7", "#6da7ec", "#1c5cab"]
PALETA_IDIOMAS = ["#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#4a3aa7", "#898781"]
COLOR_AUSENCIA = "#898781"

_PLANTILLA = """<!doctype html>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>__TITULO__</title>
<style>
  :root{
    --surface:#fcfcfb; --plane:#f9f9f7; --ink:#0b0b0b; --ink-2:#52514e;
    --muted:#898781; --grid:#e1e0d9; --line:#c3c2b7;
    --borde:rgba(11,11,11,.10);
  }
  *{box-sizing:border-box}
  body{margin:0;background:var(--plane);color:var(--ink);
       font:14px/1.5 system-ui,-apple-system,"Segoe UI",sans-serif}
  .wrap{max-width:1180px;margin:0 auto;padding:24px 16px 64px}
  h1{font-size:20px;margin:0 0 4px}
  h2{font-size:15px;margin:0 0 2px;font-weight:600}
  .sub{color:var(--ink-2);font-size:13px;margin:0 0 20px}
  .nota{color:var(--muted);font-size:12px;margin:4px 0 0}
  .panel{background:var(--surface);border:1px solid var(--borde);border-radius:10px;
         padding:16px;margin:0 0 16px}
  .controles{position:sticky;top:0;z-index:5;background:var(--surface);
             border:1px solid var(--borde);border-radius:10px;padding:12px 16px;
             margin:0 0 16px;display:flex;flex-wrap:wrap;gap:18px;align-items:flex-start}
  .grupo{display:flex;flex-direction:column;gap:6px}
  .grupo > span{font-size:11px;text-transform:uppercase;letter-spacing:.04em;color:var(--muted)}
  .chips{display:flex;flex-wrap:wrap;gap:6px;max-width:620px}
  .chip{border:1px solid var(--line);background:var(--surface);color:var(--ink-2);
        border-radius:999px;padding:3px 10px;font-size:12px;cursor:pointer;
        font-variant-numeric:tabular-nums}
  .chip[aria-pressed="true"]{background:var(--ink);border-color:var(--ink);color:#fff}
  .chip small{opacity:.65;margin-left:4px}
  .chip.sin-idioma{font-style:italic}
  label.sw{display:flex;gap:6px;align-items:center;font-size:13px;cursor:pointer}
  .aviso{background:#fff6e6;border:1px solid #f0d9a8;color:#6a4c00;
         border-radius:8px;padding:10px 14px;margin:0 0 16px;font-size:13px}
  .aviso:empty{display:none}
  .grafica{width:100%;height:300px}
  .trio{display:grid;grid-template-columns:repeat(3,1fr);gap:12px}
  @media(max-width:820px){.trio{grid-template-columns:1fr}}
  table{border-collapse:collapse;font-size:12px;width:100%;
        font-variant-numeric:tabular-nums}
  th,td{text-align:right;padding:4px 8px;border-bottom:1px solid var(--grid)}
  th:first-child,td:first-child{text-align:left}
  .fallo{padding:40px;text-align:center;color:var(--ink-2)}
</style>
<div class="wrap">
  <h1>__TITULO__</h1>
  <p class="sub">__SUBTITULO__</p>
  <div id="sinlib" class="fallo" hidden>
    No se pudo cargar la biblioteca de gráficas desde el CDN.<br>
    La página necesita conexión a <code>cdnjs.cloudflare.com</code>. Los datos
    están embebidos y siguen en la tabla de cada panel.
  </div>
  <div id="app" hidden>
    <div class="controles">
      <div class="grupo"><span>Idioma</span><div class="chips" id="f-idioma"></div></div>
      <div class="grupo"><span>Medida</span>
        <label class="sw"><input type="radio" name="modo" value="prop" checked> Proporción</label>
        <label class="sw"><input type="radio" name="modo" value="abs"> Conteo absoluto</label>
      </div>
      <div class="grupo"><span>Opciones</span>
        <label class="sw"><input type="checkbox" id="suav"> Media móvil __VENTANA__ días</label>
        <label class="sw"><input type="checkbox" id="auto"> Sólo tuits autosuficientes</label>
      </div>
    </div>
    <div id="aviso" class="aviso"></div>
    <div class="panel">
      <h2>Volumen diario del corpus</h2>
      <p class="nota">Tuits recogidos por día, apilados por idioma. Es el
         denominador de todo lo demás.</p>
      <div id="g-volumen" class="grafica"></div>
    </div>
    <div class="panel">
      <h2>Saliencia dimensional</h2>
      <p class="nota">Proporción diaria de tuits que <em>tocan</em> cada dimensión,
         es decir que recibieron un nivel de valencia en vez de ausencia. Con un
         corpus de líneas de tiempo completas, esta es la medida principal.</p>
      <div id="g-saliencia" class="grafica"></div>
    </div>
    <div id="series"></div>
    <div class="panel">
      <h2>Distribución de valencia · todo el corpus</h2>
      <p class="nota">Sólo calificaciones con valencia; la ausencia queda fuera
         del apilado y se reporta aparte.</p>
      <div id="g-global" class="grafica" style="height:220px"></div>
      <div id="t-global"></div>
    </div>
    <div class="panel">
      <h2>Distribución de valencia por período</h2>
      <p class="nota">Misma escala y mismo orden de niveles en los tres, para que
         las diferencias se lean como diferencias.</p>
      <div class="trio" id="g-periodos"></div>
    </div>
    <p class="nota">__LIMITACION__</p>
  </div>
</div>
<script src="https://cdnjs.cloudflare.com/ajax/libs/echarts/5.5.1/echarts.min.js"></script>
<script>
const D = __DATOS__;
const PAL = __PAL_VAL__, PAL_IDIOMA = __PAL_IDIOMA__, C_AUS = "__C_AUS__";
const INK="#0b0b0b", INK2="#52514e", MUTED="#898781", GRID="#e1e0d9", SURF="#fcfcfb";

if (!window.echarts) { document.getElementById("sinlib").hidden = false; }
else {
document.getElementById("app").hidden = false;

const sel = new Set(D.idiomas.map(x => x.codigo));
let modo = "prop", suavizar = false, soloAuto = false;

// ── Controles ──────────────────────────────────────────────────────────
const cont = document.getElementById("f-idioma");
D.idiomas.forEach(x => {
  const b = document.createElement("button");
  b.className = "chip" + (x.sin_idioma ? " sin-idioma" : "");
  b.setAttribute("aria-pressed", "true");
  b.innerHTML = x.codigo + " <small>" + x.n.toLocaleString("es") + "</small>";
  if (x.sin_idioma) b.title = "No es un idioma: la API lo usa cuando no hay texto del cual inferirlo";
  b.onclick = () => {
    if (sel.has(x.codigo)) { sel.delete(x.codigo); b.setAttribute("aria-pressed","false"); }
    else { sel.add(x.codigo); b.setAttribute("aria-pressed","true"); }
    pintar();
  };
  cont.appendChild(b);
});
document.querySelectorAll('input[name=modo]').forEach(r =>
  r.onchange = e => { modo = e.target.value; pintar(); });
document.getElementById("suav").onchange = e => { suavizar = e.target.checked; pintar(); };
document.getElementById("auto").onchange = e => { soloAuto = e.target.checked; pintar(); };

// ── Reducciones sobre el agregado ──────────────────────────────────────
const pasa = r => sel.has(r.idioma) && (!soloAuto || r.autosuficiente === true);
const fechas = D.fechas;
const idx = Object.fromEntries(fechas.map((f, i) => [f, i]));

function media(serie, k) {                    // media móvil centrada, ignora huecos
  if (!suavizar) return serie;
  const m = Math.floor(k / 2);
  return serie.map((_, i) => {
    let s = 0, n = 0;
    for (let j = Math.max(0, i - m); j <= Math.min(serie.length - 1, i + m); j++)
      if (serie[j] !== null) { s += serie[j]; n++; }
    return n ? s / n : null;
  });
}

function porCriterio(slug) {
  const niv = D.criterios.find(c => c.slug === slug).niveles;
  const val = Object.fromEntries(niv.map(n => [n, fechas.map(() => 0)]));
  const conVal = fechas.map(() => 0);   // calificaciones con valencia
  const aus = fechas.map(() => 0);      // ausencia: nivel 0 o no aplicable
  const fal = fechas.map(() => 0);      // fallidas: no sabemos, no son ninguna de las dos
  const hay = fechas.map(() => false);
  D.agg.filter(r => r.slug === slug && pasa(r)).forEach(r => {
    const i = idx[r.fecha_str]; if (i === undefined) return;
    hay[i] = true;
    if (r.clase === "valencia" && val[r.nivel]) { val[r.nivel][i] += r.n; conVal[i] += r.n; }
    else if (r.clase === "ausencia") aus[i] += r.n;
    else fal[i] += r.n;
  });
  // Denominador de la saliencia: valencia + ausencia. Las fallidas quedan fuera
  // de ambos lados — una calificación que reventó no dice que el tuit tocara la
  // dimensión ni que no la tocara, y meterla en el denominador la contaría como
  // ausencia, o en el numerador como saliencia. Se reportan aparte.
  const base = fechas.map((_, i) => conVal[i] + aus[i]);
  return { niv, val, conVal, aus, fal, base, hay };
}

function ejeY() {
  return { type: "value", axisLabel: { color: MUTED,
             formatter: modo === "prop" ? (v => Math.round(v * 100) + "%") : undefined },
           splitLine: { lineStyle: { color: GRID } },
           max: modo === "prop" ? 1 : null };
}
const ejeX = () => ({ type: "category", data: fechas, boundaryGap: false,
  axisLabel: { color: MUTED, formatter: v => v.slice(5) },
  axisLine: { lineStyle: { color: "#c3c2b7" } } });

const bandas = () => ({ silent: true, symbol: "none", data: D.cortes.map((p, i) => ([
  { xAxis: p.desde, itemStyle: { color: i % 2 ? "rgba(11,11,11,.035)" : "rgba(11,11,11,.015)" },
    label: { show: true, formatter: p.nombre, color: MUTED, fontSize: 10,
             position: "insideTop", rotate: 0 } },
  { xAxis: p.hasta || fechas[fechas.length - 1] } ])) });

const baseOpt = () => ({
  backgroundColor: "transparent",
  grid: { left: 52, right: 16, top: 28, bottom: 34 },
  tooltip: { trigger: "axis", axisPointer: { type: "cross", label: { show: false } },
             valueFormatter: v => v === null ? "—"
               : (modo === "prop" ? (v * 100).toFixed(1) + "%" : Math.round(v)) },
  legend: { top: 0, textStyle: { color: INK2, fontSize: 11 }, itemHeight: 8, itemWidth: 12 },
});

// ── Gráficas ───────────────────────────────────────────────────────────
const gVol = echarts.init(document.getElementById("g-volumen"));
const gSal = echarts.init(document.getElementById("g-saliencia"));
const gGlo = echarts.init(document.getElementById("g-global"));
const gSerie = {}, gPer = {};

D.criterios.forEach(c => {
  const d = document.createElement("div");
  d.className = "panel";
  d.innerHTML = '<h2>' + c.id + ' · ' + c.nombre + '</h2>' +
    '<p class="nota">Valencia diaria, apilada del más negativo al más positivo. ' +
    'La ausencia (nivel ' + (c.ausencia.join(", ") || "—") + ') no entra en el apilado.</p>' +
    '<div class="grafica" id="g-' + c.slug + '"></div>';
  document.getElementById("series").appendChild(d);
  gSerie[c.slug] = echarts.init(d.querySelector(".grafica"));

  const p = document.createElement("div");
  p.className = "panel";
  p.style.margin = "0";
  p.innerHTML = '<h2 style="font-size:13px">' + c.id + '</h2><div class="grafica" ' +
    'id="p-' + c.slug + '" style="height:200px"></div>';
  document.getElementById("g-periodos").appendChild(p);
  gPer[c.slug] = echarts.init(p.querySelector(".grafica"));
});

function pintar() {
  const nSel = D.idiomas.filter(x => sel.has(x.codigo)).reduce((a, b) => a + b.n, 0);
  const av = document.getElementById("aviso");
  if (sel.size === 0) av.textContent = "Ningún idioma seleccionado: no hay datos que mostrar.";
  else if (nSel < D.umbral_muestra)
    av.textContent = "La selección deja " + nSel.toLocaleString("es") + " tuits, por debajo de "
      + D.umbral_muestra + ". A esa densidad la serie diaria describe unos pocos tuits, no una tendencia.";
  else av.textContent = "";

  // Volumen por idioma
  const vol = {};
  D.volumen.filter(pasa).forEach(r => {
    (vol[r.idioma] = vol[r.idioma] || fechas.map(() => 0))[idx[r.fecha_str]] += r.n;
  });
  const idiomasOrden = D.idiomas.map(x => x.codigo).filter(c => vol[c]);
  gVol.setOption({ ...baseOpt(), xAxis: ejeX(),
    yAxis: { type: "value", axisLabel: { color: MUTED }, splitLine: { lineStyle: { color: GRID } } },
    series: idiomasOrden.map((c, i) => ({
      name: c, type: "bar", stack: "v", data: vol[c],
      itemStyle: { color: PAL_IDIOMA[i % PAL_IDIOMA.length],
                   borderColor: SURF, borderWidth: 2 } })),
    markArea: undefined }, true);

  // Saliencia: proporción que toca cada dimensión
  gSal.setOption({ ...baseOpt(), xAxis: ejeX(),
    yAxis: { type: "value", max: modo === "prop" ? null : null,
             axisLabel: { color: MUTED, formatter: v => Math.round(v * 100) + "%" },
             splitLine: { lineStyle: { color: GRID } } },
    tooltip: { trigger: "axis", valueFormatter: v => v === null ? "—" : (v * 100).toFixed(1) + "%" },
    series: D.criterios.map((c, i) => {
      const { conVal, base, hay } = porCriterio(c.slug);
      const s = fechas.map((_, j) => hay[j] && base[j] ? conVal[j] / base[j] : null);
      return { name: c.id, type: "line", smooth: false, connectNulls: false,
               symbolSize: 6, lineStyle: { width: 2 }, data: media(s, D.ventana),
               itemStyle: { color: PAL_IDIOMA[i % PAL_IDIOMA.length] },
               markArea: i === 0 ? bandas() : undefined };
    }) }, true);

  // Serie diaria por criterio
  D.criterios.forEach(c => {
    const { niv, val, hay } = porCriterio(c.slug);
    const den = fechas.map((_, j) => niv.reduce((a, n) => a + val[n][j], 0));
    gSerie[c.slug].setOption({ ...baseOpt(), xAxis: ejeX(), yAxis: ejeY(),
      series: niv.map((n, i) => ({
        name: n, type: "bar", stack: "s",
        data: fechas.map((_, j) => !hay[j] || (modo === "prop" && !den[j]) ? null
          : (modo === "prop" ? val[n][j] / den[j] : val[n][j])),
        itemStyle: { color: PAL[i % PAL.length], borderColor: SURF, borderWidth: 2 },
        markArea: i === 0 ? bandas() : undefined })) }, true);
  });

  // Distribución global y por período
  const barra = (filtroFechas) => D.criterios.map(c => {
    const { niv, val } = porCriterio(c.slug);
    const suma = niv.map(n => fechas.reduce((a, f, j) =>
      a + (filtroFechas(f) ? val[n][j] : 0), 0));
    const t = suma.reduce((a, b) => a + b, 0);
    return { id: c.id, niv, suma, t };
  });
  const pintaBarras = (g, datos, alto) => {
    // Unión de niveles: dos criterios de la misma rúbrica pueden declarar
    // escalas distintas, y una barra apilada necesita un vocabulario común.
    const niv = [];
    D.criterios.forEach(c => c.niveles.forEach(n => { if (!niv.includes(n)) niv.push(n); }));
    g.setOption({ ...baseOpt(),
      grid: { left: 78, right: 16, top: 28, bottom: 24 },
      xAxis: { type: "value", max: modo === "prop" ? 1 : null,
               axisLabel: { color: MUTED, formatter: modo === "prop" ? (v => Math.round(v*100)+"%") : undefined },
               splitLine: { lineStyle: { color: GRID } } },
      yAxis: { type: "category", data: datos.map(d => d.id).reverse(),
               axisLabel: { color: INK2 }, axisLine: { lineStyle: { color: "#c3c2b7" } } },
      tooltip: { trigger: "axis", axisPointer: { type: "shadow" },
                 valueFormatter: v => modo === "prop" ? (v*100).toFixed(1)+"%" : Math.round(v) },
      series: niv.map((n, i) => ({
        name: n, type: "bar", stack: "g",
        data: datos.map(d => {
          const k = d.niv.indexOf(n);
          if (k < 0) return 0;
          return modo === "prop" ? (d.t ? d.suma[k] / d.t : 0) : d.suma[k];
        }).reverse(),
        itemStyle: { color: PAL[i % PAL.length], borderColor: SURF, borderWidth: 2 } })) }, true);
  };
  pintaBarras(gGlo, barra(() => true));
  D.criterios.forEach(c => {
    const g = gPer[c.slug];
    const niv = c.niveles;
    const filas = D.cortes.map(p => {
      const dentro = f => f >= p.desde && (!p.hasta || f < p.hasta);
      const { val } = porCriterio(c.slug);
      const suma = niv.map(n => fechas.reduce((a, f, j) => a + (dentro(f) ? val[n][j] : 0), 0));
      const t = suma.reduce((a, b) => a + b, 0);
      return { nombre: p.nombre, suma, t };
    });
    g.setOption({ ...baseOpt(),
      grid: { left: 96, right: 16, top: 28, bottom: 24 },
      xAxis: { type: "value", max: modo === "prop" ? 1 : null,
               axisLabel: { color: MUTED, fontSize: 10,
                 formatter: modo === "prop" ? (v => Math.round(v*100)+"%") : undefined },
               splitLine: { lineStyle: { color: GRID } } },
      yAxis: { type: "category", data: filas.map(f => f.nombre.split(" ")[0] + " (" + f.t + ")").reverse(),
               axisLabel: { color: INK2, fontSize: 10 } },
      tooltip: { trigger: "axis", axisPointer: { type: "shadow" },
                 valueFormatter: v => modo === "prop" ? (v*100).toFixed(1)+"%" : Math.round(v) },
      legend: { show: false },
      series: niv.map((n, i) => ({
        name: n, type: "bar", stack: "p",
        data: filas.map(f => modo === "prop" ? (f.t ? f.suma[i] / f.t : 0) : f.suma[i]).reverse(),
        itemStyle: { color: PAL[i % PAL.length], borderColor: SURF, borderWidth: 2 } })) }, true);
  });

  // Tabla de respaldo: identidad nunca queda sólo en el color
  const dg = barra(() => true);
  const nivU = [];
  D.criterios.forEach(c => c.niveles.forEach(n => { if (!nivU.includes(n)) nivU.push(n); }));
  let html = "<table><tr><th>criterio</th>" +
    nivU.map(n => "<th>" + n + "</th>").join("") +
    "<th>n</th></tr>";
  dg.forEach(d => {
    html += "<tr><td>" + d.id + "</td>" +
      nivU.map(n => { const i = d.niv.indexOf(n);
        return "<td>" + (i < 0 ? "—" : (modo === "prop"
          ? (d.t ? (d.suma[i] / d.t * 100).toFixed(1) + "%" : "—")
          : d.suma[i].toLocaleString("es"))) + "</td>"; }).join("") +
      "<td>" + d.t.toLocaleString("es") + "</td></tr>";
  });
  document.getElementById("t-global").innerHTML = html + "</table>";
}

pintar();
addEventListener("resize", () => {
  [gVol, gSal, gGlo, ...Object.values(gSerie), ...Object.values(gPer)]
    .forEach(g => g.resize());
});
}
</script>
"""


def construir_html(datos: dict = None) -> str:
    """Un solo documento para los dos destinos: lo que se ve en el Colab y lo
    que se descarga son el mismo HTML, así no pueden divergir."""
    datos = datos if datos is not None else DATOS
    p = datos["periodos"]
    sub = (f"{sum(x['tuits'] for x in p):,} tuits · {len(datos['fechas'])} días · "
           f"{len(datos['criterios'])} criterios · día cortado en {ZONA_HORARIA} · "
           + " · ".join(f"{x['nombre']}: {x['tuits']:,}" for x in p))
    lim = ("El corpus son las líneas de tiempo completas de un conjunto acotado de "
           "cuentas, no una muestra aleatoria de la conversación sobre México. Los "
           "cambios entre períodos son coincidencia temporal, no causalidad, y unas "
           "pocas cuentas aportan buena parte del volumen: un movimiento en la serie "
           "puede ser el humor de una persona.")
    return (_PLANTILLA
            .replace("__TITULO__", "Calificaciones por rúbrica")
            .replace("__SUBTITULO__", sub)
            .replace("__LIMITACION__", lim)
            .replace("__VENTANA__", str(VENTANA_SUAVIZADO))
            .replace("__PAL_VAL__", json.dumps(PALETA_VALENCIA))
            .replace("__PAL_IDIOMA__", json.dumps(PALETA_IDIOMAS))
            .replace("__C_AUS__", COLOR_AUSENCIA)
            .replace("__DATOS__", json.dumps(datos, ensure_ascii=False)))


def publicar(datos: dict = None, mostrar: bool = True, descargar: bool = True) -> Path:
    html = construir_html(datos)
    destino = Path(SALIDA_HTML)
    destino.write_text(html, encoding="utf-8")
    print(f"✅ '{destino}'  ({destino.stat().st_size/1024:.0f} KB)")

    if SALIDA_DIR_DRIVE:
        try:
            montar_drive_si_hace_falta(SALIDA_DIR_DRIVE)
            d = Path(SALIDA_DIR_DRIVE); d.mkdir(parents=True, exist_ok=True)
            (d / destino.name).write_text(html, encoding="utf-8")
            print(f"✅ '{d / destino.name}'")
        except Exception as exc:
            print(f"   (no se pudo escribir en Drive: {exc})")

    if mostrar:
        from IPython.display import HTML, display
        display(HTML(html))

    print("\n   ⓘ  Al reabrir el notebook guardado las gráficas dejan de ser")
    print("      interactivas hasta reejecutar esta celda: el .ipynb guarda el")
    print(f"      HTML, no el estado del JavaScript. El artefacto que sobrevive")
    print(f"      es '{destino}' — ábrelo aparte y conserva todos los filtros.")

    if descargar:
        try:
            from google.colab import files
            files.download(str(destino))
        except Exception:
            pass
    return destino


RUTA_HTML = publicar()


---
## § 9 · Notas de validación

Lista de comprobación antes de confiar en una corrida completa:

1. **`rubrica.json` revisado a mano.** ¿Están todos los criterios? ¿Los descriptores completos, no truncados? Es el punto de control más barato y el que más errores atrapa.
2. **Prueba de humo leída, no sólo ejecutada.** ¿Las justificaciones dicen algo específico del tweet, o son genéricas? ¿Los `aplicable=False` tienen sentido?
3. **Costo real contra estimado.** Compáralo en la consola de facturación de AI Studio después de la primera corrida y ajusta `PRECIOS` o `JUSTIF_MAX_PALABRAS` si difiere.
4. **Reanudación probada.** Interrumpe la corrida a mitad (botón de stop), reejecuta la celda y confirma que reporta resultados reutilizados en lugar de empezar de cero.
5. **Rúbrica distinta.** Carga otra rúbrica con un número diferente de criterios y confirma que nada requiere tocar código.
6. **Sesión limpia.** Reinicia el entorno de ejecución y corre de arriba a abajo: no debe quedar estado residual.

### Si algo sale mal

| Síntoma | Causa probable |
|---|---|
| `FUERA_DE_ESCALA` frecuente | las etiquetas de la rúbrica son ambiguas o muy largas; simplifícalas en `rubrica.json` |
| Muchos `LIMITE_TASA` | baja `CONCURRENCIA`, o la clave es de tier gratuito |
| Todo `NO_APLICABLE` | la rúbrica no corresponde a este tipo de texto — revisa §6 antes de gastar más |
| Un criterio 100% en un nivel | descriptor mal parseado del PDF, o consigna sesgada |
| Justificaciones genéricas | sube `JUSTIF_MAX_PALABRAS`, o considera `THINKING_BUDGET=-1` (2.7× más caro) |
| Acentos roídos en Excel | el archivo se abrió sin respetar el BOM; impórtalo indicando UTF-8 |